<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>⚡ pi-H3: Video + Audio in 2 to 4 Steps</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle GPU T4 x2 Edition | Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0;'>Text, first frame or first + last frame to video with synchronized audio · No VAE decoder | MiniMax H3 20B INT8 | Powered by Wan2GP</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-GPU%20T4%20x2-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />

  <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---

**[pi-H3](https://huggingface.co/Lakonik/pi-H3)** (Stanford / Luma AI) makes **MiniMax H3** videos with synchronized audio in only **2 or 4 steps**. In the fast modes its last step draws the frames directly, with no VAE decoder; the 🎨 modes decode them with Sol-H3-VAE, a fast distilled version of H3's VAE decoder. Write a prompt, optionally add a first frame, a last frame or both, and get an MP4 with sound.

**Quick start:** run **Step 1** and **Step 2**, open the link Step 2 prints, click the ⚡ quick test under **📚 Examples** at the bottom of the page and press **🎬 Generate**. It takes about a minute.

## ✍️ How to prompt

H3 reads structured prompts. Write the whole clip, its sound and any spoken lines:

```
integrated_multimodal_description: [Shot 1] Live-action, cinematic, a slow push-in on ... A deep narrator (S1) says off-screen: <d>[English] In a world where ...</d>

overall_soundscape: A low electrical hum, a creaking chair ...

non_diegetic_music: Ominous low brass and taiko drums build slowly.
```

- **First / last frame:** open *🖼️ First / last frame* in the app. Pictures are cropped to the chosen aspect ratio. Start the prompt with how they align, for example *"For the target video, at 0.00 seconds into the target video, &lt;Picture 1&gt; (from [Shot 1]) is fully referenced."*, then describe what happens. The examples show the full format.
- **Steps:** ⚡ 2 steps is the fastest; 💎 4 steps adds detail; 🎨 2 or 4 steps + VAE decoder give cleaner frames and textures (🎨 4 steps is the best quality).
- **Size:** 480p is the quickest on T4; 768p (1 MP) is the model's native size.

## ⏱️ How long it takes on 2 x T4

Text to video with ⚡ Fast attention:

| Video | ⚡ 2 steps | 💎 4 steps | 🎨 2 steps + VAE | 🎨 4 steps + VAE |
|---|---|---|---|---|
| 480p · 5 s | **~1 min** | ~2 min | ~1 min | ~2 min |
| 480p · 10 s | ~2.5 min | ~4 min | ~2.5 min | ~4 min |
| 576p · 5 s | ~2 min | ~3 min | ~2 min | ~3 min |
| 576p · 10 s | ~4 min | ~7.5 min | ~4 min | ~7.5 min |
| 768p · 5 s | ~3 min | ~5.5 min | ~3 min | ~5.5 min |
| 768p · 10 s | ~11 min | ~21 min | ~11 min | ~21 min |

- **First / last frame:** adds about half a minute at 480p and about a minute at 768p (the frames are encoded and also shown to the text encoder).
- **🎨 VAE decoder:** adds about 5-10 s at 480p and 10-20 s at 768p.
- **Exact attention:** about 1.5x the time at 480p · 5 s, up to 2.3x at 768p · 10 s.
- **Long clips:** 576p from 12 s and 768p from 7 s use a memory-saving mode that is about 1.4x slower per step. 768p clips longer than 12 s do not fit on a T4; pick 576p or a shorter duration.

**⚡ Attention** (next to the seed): *Fast* uses Comfy Kitchen's INT8 attention for the T4 and is up to about twice as fast; *Exact* is the full-precision reference.

> **Responsible use:** only animate pictures you have the rights to, never put a real person into a video without their agreement, and label results as AI-generated.

## 📦 Step 1: Environment, Wan2GP and model downloads

Installs Wan2GP and the packages it needs, then downloads ~48 GB of weights (about 7-10 minutes): MiniMax H3 20B in INT8, the pi-H3 adapters, the Qwen3-VL text encoder, the VAEs and the Sol-H3-VAE decoder. Files you attach as a Kaggle Dataset are used instead of downloading.

In [ ]:
# Step 1: environment, pinned Wan2GP, dependencies and model downloads
import os
import sys
import time
import shutil
import subprocess
import threading
import psutil
from concurrent.futures import ThreadPoolExecutor


def stop_old_engine():
    # A previous run's engine (or one left behind by a kernel restart) still holds both GPUs and ~24 GB of RAM.
    stale = []
    for p in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            args, name = p.info["cmdline"] or [], (p.info["name"] or "").lower()
            engine = "python" in name and any(os.path.basename(a) == "run_pih3.py" for a in args[1:3])
            if p.pid != os.getpid() and (engine or name == "cloudflared" or name.startswith("frpc")):
                p.kill()
                stale.append(p)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    psutil.wait_procs(stale, timeout=20)
    if stale:
        print(f"🧹 Stopped {len(stale)} leftover engine process(es): GPU memory and RAM are free again")


stop_old_engine()

# ---- Settings (every source is locked to one version) ----
WAN2GP_COMMIT = "38758d3c73366422efcf609abb63cab8385f3552"   # deepbeepmeep/Wan2GP, 12 Nov 2026 (the version this notebook is built on)
H3_REVISION = "da05e475e3266b30efbdc98befdf7888a069bbcc"      # DeepBeepMeep/MiniMax-H3, 22 Nov 2026 (DiT, pi-H3 adapters, text encoder, VAEs)
PIH3_SPACE_REVISION = "c8569fd3f0a2774b096c9c23437da3e5266dfaee"  # Lakonik/pi-H3 demo Space, 22 Nov 2026 (official examples)
MODELS_DIR = "/kaggle/tmp/pih3_models" if os.path.isdir("/kaggle") else os.path.abspath("pih3_models")

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"  # no second copy of 48 GB in the xet cache
os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    gpus = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip().splitlines()
except Exception:
    gpus = []
ram_gb = psutil.virtual_memory().total / 1024**3
if len(gpus) < 2:
    print("❌ This notebook needs 2 GPUs: Settings → Accelerator → GPU T4 x2")
print(f"✅ pi-H3 | {len(gpus)} x {gpus[0] if gpus else 'no GPU'} | RAM {ram_gb:.0f} GB")

# ---- 1. Wan2GP, fetched by commit (never a branch) ----
t0 = time.time()
if not os.path.isdir("Wan2GP/.git"):
    subprocess.run(["git", "init", "-q", "Wan2GP"], check=True)
    subprocess.run(["git", "-C", "Wan2GP", "remote", "add", "origin", "https://github.com/DeepBeepMeep/Wan2GP.git"], check=True)
subprocess.run(["git", "-C", "Wan2GP", "fetch", "-q", "--depth", "1", "origin", WAN2GP_COMMIT], check=True)
subprocess.run(["git", "-C", "Wan2GP", "reset", "-q", "--hard", "FETCH_HEAD"], check=True)  # also undoes stale edits on re-runs
print(f"✅ Wan2GP @ {WAN2GP_COMMIT[:10]} ({time.time() - t0:.0f}s)")

# ---- 2. Dependencies: only what the H3 path imports ----
# (Wan2GP's full requirements.txt pins an onnxruntime nightly that is not on PyPI.)
t0 = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", "--timeout", "120", "--retries", "5",
                "comfy-kitchen==0.2.35", "transformers==4.54.0", "diffusers==0.36.0", "gguf==0.17.1", "optimum-quanto==0.2.7",
                "accelerate>=1.1.1", "gradio==5.29.0", "huggingface_hub[hf_xet]", "hf_xet>=1.5.2", "sentencepiece", "soundfile",
                "ffmpeg-python", "imageio", "imageio-ffmpeg"], check=True)
print(f"✅ Dependencies installed ({time.time() - t0:.0f}s)")

# ---- 3. Model weights (parallel; files from an attached Kaggle Dataset are reused) ----
H3, TE, SPACE = "DeepBeepMeep/MiniMax-H3", "Qwen3-VL-32B-Instruct", "Lakonik/pi-H3"
BIG = [(H3, "MiniMax-H3-FL2VA-pruned_rank8_int8_convrot.safetensors", 21.06, H3_REVISION),
       (H3, f"{TE}/qwen3vl-32B-MiniMax-H3-Q4_K_M.gguf", 14.58, H3_REVISION),
       (H3, "minimax_h3/minimax_h3_video_vae_int8_convrot.safetensors", 2.81, H3_REVISION),
       (H3, "MiniMax-H3-audio_vae_fp32.safetensors", 0.61, H3_REVISION)]
for steps in (2, 4):  # the pi-H3 adapters (Stanford / Luma AI), repacked for Wan2GP's layer names
    BIG += [(H3, f"loras/minimax_h3_fl2va_pi_{steps}step_dwd_lora_bf16.safetensors", 0.57, H3_REVISION),
            (H3, f"minimax_h3_fl2va_pi_{steps}step_dwd_int8_convrot.safetensors", 1.74, H3_REVISION)]
BIG += [(H3, f"loras/minimax_h3_pi_{steps}step_lora_bf16.safetensors", 0.60, H3_REVISION) for steps in (2, 4)]  # VAE decoder modes
BIG += [(H3, "minimax_h3/minimax_h3_sol_vae_int8_convrot.safetensors", 2.00, H3_REVISION)]  # Sol-H3-VAE decoder (NVIDIA), INT8
SMALL = [(H3, f"{TE}/{name}", 0.0, H3_REVISION) for name in
         ("config.json", "tokenizer.json", "tokenizer_config.json", "vocab.json", "preprocessor_config.json", "chat_template.json")]
EXAMPLES = ["examples.json", "distracted_boyfriend.webp", "woman_cat_meme_first.webp", "woman_cat_meme_last.webp"]
os.makedirs(MODELS_DIR, exist_ok=True)

attached = {}
if os.path.isdir("/kaggle/input"):
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            attached.setdefault(f, os.path.join(root, f))


def fetch(item):
    from huggingface_hub import hf_hub_download
    repo, filename, size_gb, revision = item
    dest = os.path.join(MODELS_DIR, filename)
    if os.path.exists(dest):
        return f"✓ {filename}"
    source = attached.get(os.path.basename(filename)) if size_gb else None
    if source and os.path.getsize(source) > 0:
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        os.symlink(source, dest)
        return f"✓ {filename} (from attached dataset)"
    for attempt in range(1, 4):
        try:
            hf_hub_download(repo_id=repo, filename=filename, revision=revision, local_dir=MODELS_DIR)
            return f"✅ {filename}" + (f" ({size_gb:.1f} GB)" if size_gb else "")
        except Exception as e:
            if attempt == 3:
                raise
            say(f"⚠️ {filename}: {e} - retrying ({attempt}/3)")
            time.sleep(5 * attempt)


def fetch_example(name):
    # the pi-H3 demo's own example prompts and pictures
    from huggingface_hub import hf_hub_download
    dest = os.path.join(MODELS_DIR, "examples", name)
    if os.path.exists(dest):
        return
    path = hf_hub_download(SPACE, f"demo/examples/minimax_h3/{name}", repo_type="space", revision=PIH3_SPACE_REVISION,
                           local_dir=os.path.join(MODELS_DIR, "examples", ".hf"))
    shutil.copy(path, dest)


def folder_bytes(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            p = os.path.join(root, f)
            if not os.path.islink(p):
                try:
                    total += os.path.getsize(p)
                except OSError:
                    pass
    return total


todo = [item for item in BIG if not os.path.exists(os.path.join(MODELS_DIR, item[1])) and os.path.basename(item[1]) not in attached]
need_gb = sum(item[2] for item in todo)
free_gb = shutil.disk_usage(MODELS_DIR).free / 1024**3
if need_gb > free_gb - 2:
    raise SystemExit(f"❌ Not enough disk space: need {need_gb:.0f} GB, {free_gb:.0f} GB free.")
done = threading.Event()
print_lock = threading.Lock()
bar_shown = [False]


def fmt_time(seconds):
    seconds = int(seconds)
    return f"{seconds // 60}m{seconds % 60:02d}s" if seconds >= 60 else f"{seconds}s"


def say(message):  # a normal line printed above the live bar
    with print_lock:
        if bar_shown[0]:
            sys.stdout.write("\r" + " " * 100 + "\r")
            bar_shown[0] = False
        print(message, flush=True)


def monitor():
    # One live line: GB done / total, %, smoothed speed, ETA (bytes on disk, which hf_xet fills as it goes)
    base = folder_bytes(MODELS_DIR)
    total = max(need_gb * 1e9, 1.0)
    start, samples = time.time(), []
    while not done.wait(1.0):
        got = folder_bytes(MODELS_DIR) - base
        now = time.time()
        samples = [(t, b) for t, b in samples if now - t <= 10] + [(now, got)]
        speed = (samples[-1][1] - samples[0][1]) / max(samples[-1][0] - samples[0][0], 1e-6) if len(samples) > 1 else 0
        got = min(got, total)
        frac = got / total
        fill = int(frac * 28)
        eta = fmt_time(max(0.0, total - got) / speed) if speed > 1e6 else "--"
        line = (f"⬇️ [{'█' * fill}{'░' * (28 - fill)}] {got / 1e9:5.1f} / {total / 1e9:.1f} GB {frac * 100:3.0f}% | "
                f"{speed / 1e6:4.0f} MB/s | ETA {eta} | {fmt_time(now - start)}")
        with print_lock:
            sys.stdout.write("\r" + line.ljust(100))
            sys.stdout.flush()
            bar_shown[0] = True


threading.Thread(target=monitor, daemon=True).start()
t0 = time.time()
os.makedirs(os.path.join(MODELS_DIR, "examples"), exist_ok=True)
try:
    with ThreadPoolExecutor(max_workers=6) as pool:
        jobs = [pool.submit(fetch, item) for item in BIG + SMALL]
        examples = [pool.submit(fetch_example, name) for name in EXAMPLES]
        for job in jobs:
            job.result()
        for job in examples:
            try:
                job.result()
            except Exception as e:
                say(f"⚠️ An official example could not be downloaded ({str(e).splitlines()[0][:100]}); the app retries at start")
finally:
    done.set()
    time.sleep(1.1)
if need_gb:
    say(f"✅ {need_gb:.1f} GB downloaded in {fmt_time(time.time() - t0)} (avg {need_gb * 1e3 / max(time.time() - t0, 1):.0f} MB/s)")
shutil.rmtree(os.path.join(MODELS_DIR, ".cache"), ignore_errors=True)
shutil.rmtree(os.path.join(MODELS_DIR, "examples", ".hf"), ignore_errors=True)

# Settings for Step 2's engine (also saved to a file, so Step 2 can be re-run alone after a kernel restart)
import json
PIH3_SETTINGS = {"PIH3_MODELS_DIR": MODELS_DIR, "PIH3_EXAMPLES_DIR": os.path.join(MODELS_DIR, "examples")}
os.environ.update(PIH3_SETTINGS)
with open("pih3_settings.json", "w") as f:
    json.dump(PIH3_SETTINGS, f)
print(f"✅ All weights ready. Run Step 2 (disk free: {shutil.disk_usage(MODELS_DIR).free / 1024**3:.0f} GB)")


## 🚀 Step 2: Engine + Gradio UI

Starts the engine: model load (~4 min), then the **Cloudflare** and **Gradio** public links. Open either one. If the Cloudflare link is still warming up, it appears in the cell a minute later.

**In the app:** write the prompt (or click an example) → pick steps, size and duration → 🎬 **Generate**. The progress bar at the top of the page keeps working if the connection drops, and *📂 Load latest video* fetches the result at any time.

In [ ]:
# Step 2: Engine + branded Gradio UI
# Writes run_pih3.py, then runs it: model load (~4 min) → public links.
import codecs
import os
import re
import sys
import subprocess

RUN_SCRIPT_CODE = r'''"""pi-H3 text / image to video with audio engine for Kaggle GPU T4 x2 - AIQUEST Academy.

Step 2 of the notebook writes this file as run_pih3.py and runs it. It drives Wan2GP's MiniMax H3 FL2VA code:

* DiT: MiniMax H3 FL2VA Pruned 20B in INT8 ConvRot with the pi-H3 2-step or 4-step adapter (Stanford / Luma AI):
  rank-64 LoRAs on blocks 0-45 run live in FP16 on top of the INT8 weights, new AdaLN curves, and blocks 46-49
  replaced whole (INT8 ConvRot). No guidance pass.
* DWD (Decoding Without a Decoder): the last step's video tokens go straight to pi-H3's pixel head (the last 8 layers
  of Sol-H3-VAE's decoder, retrained), so the frames come out without the video VAE decoder.
* VAE decoder modes: pi-H3's 2 and 4-step adapters for a regular decode, and Sol-H3-VAE's decoder (NVIDIA, distilled from H3's
  36-layer VAE decoder: 24 layers on whole frames) on both T4s at once.
* Every INT8 linear runs on Comfy Kitchen's INT8 tensor-core GEMM (W8A8 with the ConvRot Hadamard rotation).
* Tensor-parallel DiT: every block is split in half across the two T4s (attention heads and FFN columns), so both GPUs
  compute every block at the same time and only exchange the block's partial outputs. Nothing is streamed per step.
  Each GPU's half is queued from a CPU thread of its own, so neither GPU waits for the other's kernel launches.
* Attention: Comfy Kitchen's INT8 attention for the T4 by default, or exact PyTorch SDPA (FP16) in the UI.
* Qwen3-VL-32B (GGUF Q4_K_M), the VAE encoder (first / last frames) and the audio VAE are pinned in RAM and run by
  MMGP on GPU 1, unloaded before denoising.
"""
import collections
import copy
import faulthandler
import functools
import gc
import glob
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import threading
import time
import traceback
import urllib.request
import warnings

warnings.filterwarnings("ignore")
faulthandler.enable(file=sys.__stderr__, all_threads=True)  # a hard crash (segfault, abort) prints where it happened
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

WAN2GP_DIR = os.path.abspath("Wan2GP")
WORK_DIR = os.path.dirname(WAN2GP_DIR)
OUTPUT_DIR = os.path.join(WORK_DIR, "outputs")
MODELS_DIR = os.environ.get("PIH3_MODELS_DIR", "/kaggle/tmp/pih3_models")
EXAMPLES_DIR = os.environ.get("PIH3_EXAMPLES_DIR", os.path.join(MODELS_DIR, "examples"))
DIT_FILE = os.environ.get("PIH3_DIT_FILE", "MiniMax-H3-FL2VA-pruned_rank8_int8_convrot.safetensors")
TE_FILE = os.environ.get("PIH3_TE_FILE", "qwen3vl-32B-MiniMax-H3-Q4_K_M.gguf")
# pi-H3 adapters repacked for Wan2GP's layer names: the LoRA file (blocks 0-45, AdaLN, output head) and the module file
# (blocks 46-49 and the DWD pixel head, INT8 ConvRot), one pair per step count
PI_FILES = {steps: (os.path.join("loras", f"minimax_h3_fl2va_pi_{steps}step_dwd_lora_bf16.safetensors"),
                    f"minimax_h3_fl2va_pi_{steps}step_dwd_int8_convrot.safetensors") for steps in (2, 4)}
PI_VAE_LORAS = {steps: os.path.join("loras", f"minimax_h3_pi_{steps}step_lora_bf16.safetensors") for steps in (2, 4)}  # for a regular decode
SOL_VAE_FILE = os.path.join("minimax_h3", "minimax_h3_sol_vae_int8_convrot.safetensors")  # Sol-H3-VAE decoder, INT8 ConvRot
SERVER_PORT = 7860
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ.setdefault("GRADIO_TEMP_DIR", os.path.join(OUTPUT_DIR, ".gradio_cache"))
if WAN2GP_DIR not in sys.path:
    sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)

import numpy as np
import psutil
import torch
from PIL import Image

import gradio as gr


def _gb(n):
    return f"{n / 1024**3:.1f} GB"


def _ram_in_use():
    """total - available: unlike psutil's 'used', this counts pinned (page-locked) memory, which Linux files as shared."""
    vm = psutil.virtual_memory()
    return f"{_gb(vm.total - vm.available)} / {_gb(vm.total)}"


def _fmt_secs(seconds):
    seconds = int(round(seconds))
    return f"{seconds // 60}m{seconds % 60:02d}s" if seconds >= 60 else f"{seconds}s"


class ConsoleBar:
    """One live progress line per phase, redrawn every second with a carriage return (Kaggle's cell output overwrites it in place).
    A finished phase just clears its line (minimal logs: callers print one summary line where it matters). Step-based phases (denoising) interpolate inside the current
    step from the previous step's duration, so the bar keeps moving and the ETA counts down between callbacks."""

    WIDTH, LINE = 24, 104
    WATCHED = {"decode": 1800, "stitch": 600, "audio": 600, "save": 600}  # phase -> seconds before a stack dump

    def __init__(self, stream):
        self.out = stream
        self.lock = threading.RLock()
        self.phase = None
        self.visible = False
        self.drawn = 0  # characters of the live line on screen
        self.text_pending = False  # another writer is in the middle of a line: don't draw over it
        threading.Thread(target=self._loop, daemon=True).start()

    def start(self, key, label, total=None, unit=""):
        with self.lock:
            if self.phase is not None and self.phase["key"] == key:
                return
            self.finish()
            now = time.time()
            self.phase = dict(key=key, label=label, total=total, unit=unit, done=0, t0=now, t_done=now, dt=None)
            if key in self.WATCHED:  # a phase that should take seconds: dump every thread's stack if it hangs
                faulthandler.dump_traceback_later(self.WATCHED[key], file=sys.__stderr__)
            self._draw()

    def update(self, done, total=None):
        with self.lock:
            p = self.phase
            if p is None:
                return
            now = time.time()
            if total:
                p["total"] = total
            if done > p["done"]:
                p["dt"] = (now - p["t_done"]) / (done - p["done"])
                p["t_done"], p["done"] = now, done
            self._draw()

    def finish(self):
        with self.lock:
            p, self.phase = self.phase, None
            if p is None:
                return
            faulthandler.cancel_dump_traceback_later()
            self._clear()
            self.out.flush()

    def abort(self):
        """End the live line of a phase that failed (no ✅ summary)."""
        with self.lock:
            p, self.phase = self.phase, None
            if p is None:
                return
            faulthandler.cancel_dump_traceback_later()
            self._clear()
            self.out.write(f"  ❌ {p['label']} failed after {_fmt_secs(time.time() - p['t0'])}\n")
            self.out.flush()

    def _render(self):
        p, now = self.phase, time.time()
        elapsed = now - p["t0"]
        if not p["total"]:
            spinner = "⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏"[int(now * 4) % 10]
            return f"  {spinner} {p['label']} | {_fmt_secs(elapsed)}"
        progress = p["done"]
        if p["dt"] and p["done"] < p["total"]:
            progress += min(0.97, (now - p["t_done"]) / p["dt"])
        frac = min(1.0, progress / p["total"])
        fill = int(round(frac * self.WIDTH))
        line = (f"  ⏳ {p['label']:<16} [{'█' * fill}{'░' * (self.WIDTH - fill)}] {frac * 100:3.0f}% | "
                f"{p['done']}/{p['total']} {p['unit']}")
        if p["dt"]:
            eta = max(0.0, (p["total"] - p["done"]) * p["dt"] - (now - p["t_done"]))
            line += f" | {p['dt']:.1f}s each | ETA {_fmt_secs(eta)}"
        return line + f" | {_fmt_secs(elapsed)}"

    def _draw(self):
        if self.phase is None or self.text_pending:
            return
        text = self._render()
        self.drawn = max(self.drawn, len(text), self.LINE)
        self.out.write("\r" + text.ljust(self.drawn))
        self.out.flush()
        self.visible = True

    def _clear(self):
        if self.visible:
            self.out.write("\r" + " " * self.drawn + "\r")
            self.visible = False
            self.drawn = 0

    def _loop(self):
        while True:
            time.sleep(1.0)
            with self.lock:
                self._draw()


class _BarAwareStream:
    """stdout / stderr wrapper: ordinary log lines clear the live bar first and print above it."""

    def __init__(self, stream, bar):
        self._stream, self._bar = stream, bar

    def write(self, data):
        with self._bar.lock:
            if data:
                self._bar._clear()
                self._bar.text_pending = not data.endswith("\n")
            return self._stream.write(data)

    def flush(self):
        self._stream.flush()

    def __getattr__(self, name):
        return getattr(self._stream, name)


BAR = ConsoleBar(sys.stdout)
sys.stdout = _BarAwareStream(sys.stdout, BAR)
sys.stderr = _BarAwareStream(sys.stderr, BAR)



if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise SystemExit("❌ This notebook needs Kaggle's 'GPU T4 x2' accelerator (Settings → Accelerator → GPU T4 x2).")

DIT_DEV = torch.device("cuda", 0)   # one half of every DiT block
MAIN_DEV = torch.device("cuda", 1)  # the other half + text encoder + VAEs + sampler state
torch.cuda.set_device(MAIN_DEV)     # Wan2GP / MMGP use the current device for "cuda"
DEVICES = (DIT_DEV, MAIN_DEV)

torch.backends.cuda.enable_flash_sdp(False)       # Turing (SM 7.5) has no flash kernel
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
torch.backends.cudnn.benchmark = False

_props = torch.cuda.get_device_properties(0)


def _on(dev):
    """torch.cuda.device(dev) for a CUDA device; nothing for the CPU."""
    import contextlib
    return torch.cuda.device(dev) if dev.type == "cuda" else contextlib.nullcontext()


# Turing has no BF16 convolution kernels. Convolutions that reach BF16 run in FP32.
if _props.major < 8:
    def _conv_fp32_on_turing(original):
        def forward(self, input, weight, bias):
            if input.is_cuda and input.dtype == torch.bfloat16:
                bias32 = None if bias is None else bias.float()
                return original(self, input.float(), weight.float(), bias32).to(torch.bfloat16)
            return original(self, input, weight, bias)
        return forward
    for _conv in (torch.nn.Conv1d, torch.nn.Conv2d, torch.nn.Conv3d):
        _conv._conv_forward = _conv_fp32_on_turing(_conv._conv_forward)

# ==== Wan2GP runtime: quantization handlers + INT8 kernels (what wgp.py does at import) ====
# Wan2GP's shared utilities import rembg (background removal, which pulls in onnxruntime) at module level. This
# engine never removes backgrounds, so a stand-in module is registered when rembg is not installed.
import types as _types

try:
    import rembg  # noqa: F401
except Exception:
    _rembg = _types.ModuleType("rembg")

    def _no_rembg(*_args, **_kwargs):
        raise RuntimeError("Background removal (rembg) is not installed in this notebook")

    _rembg.remove = _rembg.new_session = _no_rembg
    sys.modules["rembg"] = _rembg

from mmgp import offload, quant_router

quant_router.unregister_handler(".fp8_quanto_bridge")
for _handler in ("shared.qtypes.int8_convrot", "shared.qtypes.gguf"):  # DiT, Qwen3-VL GGUF
    quant_router.register_handler(_handler)
from shared.qtypes import gguf as _gguf_handler

quant_router.register_file_extension("gguf", _gguf_handler)

from shared.kernels import int8_backend

try:
    int8_backend.configure("auto", 1)
except Exception as e:
    print(f"⚠️ INT8 kernels unavailable, using the PyTorch path: {e}")
# Kitchen splits each INT8 GEMM into row tiles that fit a 16 MB scratch buffer, which on a T4 means 64-row GEMMs
# for the 28672-wide FFN. 128 MB gives ~700-row tiles: far better tensor-core use for ~100 MB of extra scratch.
int8_backend._SCRATCH_BYTES = 128 * 1024 * 1024
INT8_LABEL = {"kitchen": "Comfy Kitchen INT8 tensor cores", "wgpkernels": "WGPKernels INT8", "triton": "Triton INT8 kernels"}.get(
    int8_backend._backend, "PyTorch INT8 fallback (slow)")
offload.shared_state["_attention"] = "sdpa"

from shared.utils import files_locator as fl

fl.set_checkpoints_paths([MODELS_DIR, "ckpts", "."])

import models.minimax_h3.minimax_h3_main as h3_main
from models.minimax_h3 import pipeline as h3_pipeline
from models.minimax_h3.minimax_h3_handler import family_handler
from models.minimax_h3.minimax_h3_main import VIDEO_VAE_INT8_FILE

AUDIO_RATE = h3_pipeline.AUDIO_SAMPLE_RATE

# ComfyUI-layout pruned checkpoints may also carry the AdaLN basis / mean they were compressed with. Wan2GP's model has
# no such tensors, so drop them before loading.
_orig_strip_wrappers = h3_main._strip_wrappers


def _strip_wrappers(state_dict, *args, **kwargs):
    for key in [k for k in state_dict if k.endswith(("adaln_basis", "adaln_mean"))]:
        state_dict.pop(key)
    return _orig_strip_wrappers(state_dict, *args, **kwargs)


h3_main._strip_wrappers = _strip_wrappers


# ==== Text encoder compute dtype ====
# Without llama.cpp kernels (no wheel for Kaggle's Python), each GGUF linear is dequantized to a dense weight and run
# as a normal matmul. Turing has no BF16 tensor cores, so a BF16 GEMM can be slower than plain FP32. Measure both on
# the real layer shape and keep the faster one (FP32 is also the more precise of the two).
def _gemm_seconds(dtype, m=512, k=5120, n=25600, reps=3):
    a = torch.randn(m, k, device=MAIN_DEV, dtype=dtype)
    b = torch.randn(n, k, device=MAIN_DEV, dtype=dtype)
    torch.nn.functional.linear(a, b)
    torch.cuda.synchronize(MAIN_DEV)
    t = time.time()
    for _ in range(reps):
        torch.nn.functional.linear(a, b)
    torch.cuda.synchronize(MAIN_DEV)
    return (time.time() - t) / reps


try:
    _bf16_s, _fp32_s = _gemm_seconds(torch.bfloat16), _gemm_seconds(torch.float32)
    TE_DTYPE = torch.float32 if _fp32_s < _bf16_s else torch.bfloat16
except Exception:
    TE_DTYPE = torch.bfloat16
torch.cuda.empty_cache()

# The text encoder's GEMMs run on the T4's FP16 tensor cores (several times FP32's speed): each GGUF weight is
# dequantized straight to FP16, and each input is scaled to a max of 32 first (per call, on the GPU), so no product
# can overflow FP16's range; the result is scaled back in the encoder's own dtype. A startup check compares it with
# the plain path on a prompt and keeps the plain path if they differ.
TE_FP16 = {"on": os.environ.get("PIH3_TE_FP16", "1") == "1", "cpu": False}
_orig_gguf_linear = _gguf_handler.GGUFWeightTensor.linear


def _gguf_linear_fp16(self, input, bias=None):
    if not (TE_FP16["on"] and torch.is_tensor(input) and (input.is_cuda or TE_FP16["cpu"])
            and input.dtype in (torch.float32, torch.bfloat16) and len(self._tensor_shape) == 2):
        return _orig_gguf_linear(self, input, bias)
    weight = self.dequantize(dtype=torch.float16, device=input.device)
    x = input.reshape(-1, input.shape[-1])
    scale = x.abs().amax().float().clamp_min(1e-30) / 32.0
    y = torch.nn.functional.linear((x.float() / scale).to(torch.float16), weight)
    del weight
    y = y.float().mul_(scale)
    if bias is not None:
        y += bias.float()
    return y.to(input.dtype).reshape(*input.shape[:-1], y.shape[-1])


_gguf_handler.GGUFWeightTensor.linear = _gguf_linear_fp16
_orig_load_text_encoder = h3_main._load_text_encoder
h3_main._load_text_encoder = lambda filename, dtype, distilled=False: _orig_load_text_encoder(filename, TE_DTYPE, distilled)
h3_main._load_latent_upscaler = lambda filename: None  # pi-H3 samples in one pass: the 2-pass latent upscaler is never used

# ==== Load the model ====
MODEL_TYPE = "minimax_h3_fl2va_pruned"
_t_load = time.time()
BAR.start("load", "Loading models (~3 min)")
model_def = {"architecture": MODEL_TYPE}
model_def.update(family_handler.query_model_def(MODEL_TYPE, model_def))
model_def["qkv_splitting"] = False             # keep the fused QKV GEMM: bigger tiles, and the tensor-parallel split below
model_def["video_vae_file"] = VIDEO_VAE_INT8_FILE  # its encoder encodes the first / last frames (the decoder is never used)
pipeline, _ = family_handler.load_model(
    model_filename=[os.path.join(MODELS_DIR, DIT_FILE)],
    model_type=MODEL_TYPE,
    base_model_type=MODEL_TYPE,
    model_def=model_def,
    dtype=torch.bfloat16,
    VAE_dtype=torch.float32,
    text_encoder_filename=os.path.join(MODELS_DIR, "Qwen3-VL-32B-Instruct", TE_FILE),
)
transformer = pipeline.transformer
transformer.cache = None
if not getattr(transformer, "use_adaln_curves", False) or transformer.adaln_t_table.shape[1] != 8:
    raise SystemExit("❌ The DiT did not load as MiniMax H3 FL2VA Pruned (rank-8 AdaLN curves)")

# Modules H3 pins to FP32 (patch projections, output heads, AdaLN curve projections) are converted here,
# since this transformer is not handed to MMGP's profile (which would otherwise apply the locks).
for _module in transformer.modules():
    _lock = getattr(_module, "_lock_dtype", None)
    if _lock is not None and isinstance(_module, torch.nn.Linear) and _module.weight.dtype.is_floating_point:
        _module.to(_lock)

# ==== Tensor-parallel DiT: each block is split in half across the two T4s ====
# Within a block, GPU 0 computes attention heads 0-27 and FFN columns 0-7167, GPU 1 the other half. Both start from
# the same residual stream (kept as a replica on each GPU), and after attention and after the FFN each GPU sends its
# partial output projection to the other, so both GPUs hold the full sum. The weight slices follow the ConvRot
# layout exactly: QKV / FFN-up are split by output rows (Q, K, V are grouped), output / FFN-down by input columns on
# 256-wide rotation-group boundaries, so every INT8 GEMM computes precisely its share of the full product.
import models.minimax_h3.transformer as _h3_transformer
from models.minimax_h3.transformer import MLP as H3MLP, Attention as H3Attention, _gated_residual, _norm_modulate, _take
from shared.qtypes.int8_convrot import _rotate_activation

_tp_cache = {"rope": None, "rope_copy": None, "temb": None, "temb_copy": None, "src": None, "replica": None}


# >>> section:tp
class Int8Shard(torch.nn.Module):
    """One GPU's slice of an INT8 ConvRot linear (a set of output rows, or a range of input columns)."""

    def __init__(self, linear, device, rows=None, cols=None):
        super().__init__()
        data, scale = linear.weight._data, linear.weight._scale.reshape(-1)
        if rows is not None:
            data, scale = data.index_select(0, rows), scale.index_select(0, rows)
        if cols is not None:
            data = data[:, cols]
        # copy=True: own storage even on the same device (never a view into the memory-mapped checkpoint)
        self.register_buffer("qdata", data.contiguous().to(device, copy=True), persistent=False)
        self.register_buffer("qscale", scale.float().contiguous().to(device, copy=True), persistent=False)
        self.group = int(getattr(linear, "_convrot_group_size", 0) or 0)
        self.out_features, self.in_features = self.qdata.shape

    def forward(self, x):
        x2 = x.reshape(-1, self.in_features)
        if int8_backend.kitchen_enabled() and self.group == 256 and x2.is_cuda and self.out_features % 4 == 0:
            out = int8_backend._linear_impl(x2, self.qdata, self.qscale, None, True)
        else:  # reference path (no Kitchen): rotate, dequantize, matmul
            weight = self.qdata.to(x2.dtype) * self.qscale.to(x2.dtype)[:, None]
            out = torch.nn.functional.linear(_rotate_activation(x2, self.group), weight)
        return out.reshape(*x.shape[:-1], self.out_features)


def _copy_to(value, dev, slot):
    """Per-forward copies of temb / rope for GPU 0 (they are the same tensor object for all 50 blocks)."""
    if value is None:
        return None
    first = value[0] if isinstance(value, tuple) else value
    if first.device == dev:
        return value
    if _tp_cache[slot] is not first:
        _tp_cache[slot] = first
        _tp_cache[slot + "_copy"] = tuple(v.to(dev) for v in value) if isinstance(value, tuple) else value.to(dev)
    return _tp_cache[slot + "_copy"]


def _all_reduce_residual(hiddens, partials, gates, segments):
    """hidden += gate * (partial_0 + partial_1) on both GPUs."""
    incoming = [partials[1].to(DEVICES[0], non_blocking=True, copy=True),
                partials[0].to(DEVICES[1], non_blocking=True, copy=True)]
    out = []
    for s, dev in enumerate(DEVICES):
        with _on(dev):
            branch = partials[s].add_(incoming[s])
            out.append(_gated_residual([hiddens[s]], [gates[s]], [branch], segments))
    return out


# Long-sequence mode: the same block math on row slices. A normal block builds several full-length intermediates per GPU
# (modulated input, fused QKV, BF16 and FP16 copies of Q/K/V, the output-projection input, the exchanged partial sums):
# roughly 75-110 KB per token. For very long sequences that does not fit next to half of the model, so in this mode only
# what attention needs at full length is kept (the residual stream and FP16 Q/K/V, ~40 KB per token). Norms,
# projections, RoPE, the FFN and the GPU exchange run on 8192-row slices. Same arithmetic, a little more Python per block.
LONG_VIDEO = {"on": False, "rows": 8192}


def _row_chunks(n):
    step = LONG_VIDEO["rows"]
    return [(a, min(n, a + step)) for a in range(0, n, step)]


def _slice_segments(segments, a, b):
    """(start, stop, modality row) segments of the packed sequence, clipped to rows [a, b) and shifted to 0."""
    return [(max(s0, a) - a, min(s1, b) - a, row) for s0, s1, row in segments if min(s1, b) > max(s0, a)]


def _apply_rope(query, key, rope):
    """H3's split-half rotary embedding on [1, n, heads, dim] tensors, in place."""
    pairs = rope.shape[-2]
    cosine, sine = rope[..., 0], rope[..., 1]
    scratch = torch.empty_like(query[..., :pairs])
    for tensor in (query, key):
        first, second = tensor[..., :pairs], tensor[..., pairs:2 * pairs]
        scratch.copy_(first)
        first.mul_(cosine).addcmul_(second, sine, value=-1)
        second.mul_(cosine).addcmul_(scratch, sine)


def _sliced_attention(shard, hidden, mod, segments, rope):
    attn = shard.attn
    rows, heads, dim = hidden.shape[0], attn.heads, attn.head_dim
    inner = heads * dim
    if isinstance(rope, tuple):
        rope = rope[0]
    q, k, v = (torch.empty(1, rows, heads, dim, device=hidden.device, dtype=torch.float16) for _ in range(3))
    for a, b in _row_chunks(rows):
        h = _norm_modulate(shard.norm1, hidden[a:b], [mod[0], mod[1]], _slice_segments(segments, a, b))
        qkv = attn.qkv_proj(h)
        del h
        query, key, value = (t.reshape(1, b - a, heads, dim) for t in qkv.split(inner, dim=-1))
        query, key = attn.q_norm(query), attn.k_norm(key)
        if rope is not None:
            _apply_rope(query, key, rope[:, a:b])
        q[:, a:b].copy_(query)
        k[:, a:b].copy_(key)
        v[:, a:b].copy_(value)
        del qkv, query, key, value
    qkv_list = [q, k, v]
    del q, k, v
    out = _h3_transformer.pay_attention(qkv_list, recycle_q=True)[0]  # [rows, heads, dim] FP16
    partial = torch.empty_like(hidden)
    for a, b in _row_chunks(rows):
        partial[a:b] = attn.out_proj(out[a:b].reshape(b - a, inner).to(hidden.dtype))
    return partial


def _sliced_mlp(shard, hidden, mod, segments):
    partial = torch.empty_like(hidden)
    for a, b in _row_chunks(hidden.shape[0]):
        h = _norm_modulate(shard.norm2, hidden[a:b], [mod[3], mod[4]], _slice_segments(segments, a, b))
        partial[a:b] = shard.mlp([h])
    return partial


def _all_reduce_sliced(hiddens, partials, gates, segments):
    """_all_reduce_residual without full-length copies: the partial sums cross between GPUs slice by slice."""
    for a, b in _row_chunks(hiddens[0].shape[0]):
        incoming = [partials[1][a:b].to(DEVICES[0], non_blocking=True, copy=True),
                    partials[0][a:b].to(DEVICES[1], non_blocking=True, copy=True)]
        for s, dev in enumerate(DEVICES):
            with _on(dev):
                partials[s][a:b].add_(incoming[s])
        del incoming
    out = []
    for s, dev in enumerate(DEVICES):
        with _on(dev):
            out.append(_gated_residual([hiddens[s]], [gates[s]], [partials[s]], segments))
    return out


# One CPU thread per GPU. With a single Python thread queuing both GPUs' kernels one after the other, anything that holds
# that thread up on one GPU (a full launch queue, a copy between the GPUs staged through host memory, an allocator flush)
# also holds back the other GPU's next kernels, and the other GPU sits idle. Each GPU has a thread of its own that queues
# its half of every block, and the two threads meet only where the math needs both halves: the two partial-sum exchanges
# per block. Same kernels in the same order on each GPU, so the result is
# bit-identical. The first transformer pass of the session runs on one thread (it compiles every kernel on both GPUs);
# the per-GPU threads take over from the second pass.
TP_THREADS = {"on": True, "warm": False}
TP_STATE = {"block": -1}  # index of the block being queued (read by the INT8 attention check)
_TP_BARRIER = threading.Barrier(len(DEVICES), timeout=1800)
_TP_POOLS = []


def _tp_pools():
    if not _TP_POOLS:
        from concurrent.futures import ThreadPoolExecutor
        for dev in DEVICES:
            init = (lambda d=dev: torch.cuda.set_device(d)) if dev.type == "cuda" else None
            _TP_POOLS.append(ThreadPoolExecutor(1, thread_name_prefix=f"tp{len(_TP_POOLS)}", initializer=init))
    return _TP_POOLS


def _tp_job(fn, s, grad, inference, args):
    try:
        with torch.inference_mode(inference), torch.set_grad_enabled(grad), _on(DEVICES[s]):
            return fn(s, *args)
    except BaseException:
        _TP_BARRIER.abort()  # the other GPU's thread stops waiting at its next exchange
        raise


def _run_on_both(fn, args_per_gpu):
    """fn(s, *args) on every GPU's own thread at once. Returns when both threads have queued their kernels, not when the
    GPUs are done with them."""
    if _TP_BARRIER.broken:
        _TP_BARRIER.reset()
    grad, inference = torch.is_grad_enabled(), torch.is_inference_mode_enabled()
    futures = [pool.submit(_tp_job, fn, s, grad, inference, args) for s, (pool, args) in enumerate(zip(_tp_pools(), args_per_gpu))]
    results, errors = [], []
    for future in futures:
        try:
            results.append(future.result())
        except BaseException as e:
            errors.append(e)
    if errors:  # the real error, not the other thread's broken barrier
        raise next((e for e in errors if not isinstance(e, threading.BrokenBarrierError)), errors[0])
    return results


def _exchange(s, hidden, partial, gate, segments, shared, sliced):
    """hidden += gate * (partial_0 + partial_1) on GPU s, from GPU s's thread. Both copies are queued (each on its source
    GPU's stream) before either thread adds into its own partial sum, so no copy can read a sum that was already changed."""
    other = 1 - s
    shared[s] = partial
    _TP_BARRIER.wait()  # both partial sums are queued
    for a, b in (_row_chunks(hidden.shape[0]) if sliced else [(0, hidden.shape[0])]):
        incoming = shared[other][a:b].to(DEVICES[s], non_blocking=True, copy=True)
        _TP_BARRIER.wait()  # both copies of these rows are queued
        partial[a:b].add_(incoming)
        del incoming
    shared[s] = None  # the other thread has queued its last read of it (before the barrier above)
    return _gated_residual([hidden], [gate], [partial], segments)


def _half_block(s, block, hidden, temb, rope, segments, shared, sliced):
    """GPU s's half of a DiT block, queued from GPU s's thread: the same kernels as TPBlock's one-thread path."""
    shard = block.shards[s]
    mod = shard.adaln_proj(temb)
    if sliced:
        partial = _sliced_attention(shard, hidden, mod, segments, rope)
    else:  # the modulated input is handed over in a list: attention frees it once Q, K and V exist
        partial = shard.attn([_norm_modulate(shard.norm1, hidden, [mod[0], mod[1]], segments)], rope=rope)
    hidden = _exchange(s, hidden, partial, mod[2], segments, shared, sliced)
    del partial
    if sliced:
        partial = _sliced_mlp(shard, hidden, mod, segments)
    else:
        partial = shard.mlp([_norm_modulate(shard.norm2, hidden, [mod[3], mod[4]], segments)])
    return _exchange(s, hidden, partial, mod[5], segments, shared, sliced)


class TPBlock(torch.nn.Module):
    def __init__(self, block, index=0):
        super().__init__()
        self.index = index
        attn, mlp = block.attn, block.mlp
        heads, head_dim = attn.heads, attn.head_dim
        inner, hidden = heads * head_dim, mlp.fc2.out_features
        ffn = mlp.fc2.in_features
        half_inner, half_ffn = inner // 2, ffn // 2
        if getattr(attn, "qkv_interleaved", False):
            raise RuntimeError("The tensor-parallel split expects grouped Q|K|V rows (quantized H3 checkpoints)")
        self.splits = []
        shards = []
        for s, dev in enumerate(DEVICES):
            h = torch.arange(s * half_inner, (s + 1) * half_inner)
            f = torch.arange(s * half_ffn, (s + 1) * half_ffn)
            qkv_rows = torch.cat((h, h + inner, h + 2 * inner))
            fc1_rows = torch.cat((f, f + ffn))  # SwiGLU: gate rows, then value rows
            out_cols = slice(s * half_inner, (s + 1) * half_inner)
            fc2_cols = slice(s * half_ffn, (s + 1) * half_ffn)
            self.splits.append({"qkv": qkv_rows, "fc1": fc1_rows, "out": out_cols, "fc2": fc2_cols})
            with torch.device("meta"):
                sattn = H3Attention(hidden, heads // 2, head_dim, attn.q_norm.eps, dtype=torch.bfloat16)
                smlp = H3MLP(hidden, half_ffn, dtype=torch.bfloat16)
            sattn.qkv_proj = Int8Shard(attn.qkv_proj, dev, rows=qkv_rows)
            sattn.out_proj = Int8Shard(attn.out_proj, dev, cols=out_cols)
            sattn.q_norm = copy.deepcopy(attn.q_norm).to(dev)
            sattn.k_norm = copy.deepcopy(attn.k_norm).to(dev)
            smlp.fc1 = Int8Shard(mlp.fc1, dev, rows=fc1_rows)
            smlp.fc2 = Int8Shard(mlp.fc2, dev, cols=fc2_cols)
            shard = torch.nn.Module()
            shard.attn, shard.mlp = sattn, smlp
            shard.norm1 = copy.deepcopy(block.norm1).to(dev)
            shard.norm2 = copy.deepcopy(block.norm2).to(dev)
            shard.adaln_proj = copy.deepcopy(block.adaln_proj).to(dev)
            shards.append(shard)
        self.shards = torch.nn.ModuleList(shards)

    @property
    def attn(self):  # H3's forward checks block.attn for q_proj / vdn
        return self.shards[1].attn

    def forward(self, x_list, temb, segments, rope, residual_signature_elements=0):
        if residual_signature_elements:
            raise RuntimeError("First Block Cache is not supported by the tensor-parallel DiT")
        hidden = _take(x_list)
        hidden = hidden if hidden.device == MAIN_DEV else hidden.to(MAIN_DEV)
        replica = _tp_cache["replica"] if _tp_cache["src"] is hidden else hidden.to(DIT_DEV, copy=True)
        hiddens = [replica, hidden]
        tembs = [_copy_to(temb, DIT_DEV, "temb"), temb]
        ropes = [_copy_to(rope, DIT_DEV, "rope"), rope]
        TP_STATE["block"] = self.index
        if TP_THREADS["on"] and TP_THREADS["warm"]:
            shared = [None] * len(DEVICES)
            hiddens = _run_on_both(_half_block, [(self, hiddens[s], tembs[s], ropes[s], segments, shared, LONG_VIDEO["on"])
                                                 for s in range(len(DEVICES))])
            _tp_cache["src"], _tp_cache["replica"] = hiddens[1], hiddens[0]
            return hiddens[1]
        if LONG_VIDEO["on"]:
            return self._forward_sliced(hiddens, tembs, ropes, segments)
        mods, partials = [], []
        for s, dev in enumerate(DEVICES):  # kernels are queued on both GPUs before either is waited for
            shard = self.shards[s]
            with _on(dev):
                mod = shard.adaln_proj(tembs[s])
                mods.append(mod)
                h = _norm_modulate(shard.norm1, hiddens[s], [mod[0], mod[1]], segments)
                partials.append(shard.attn([h], rope=ropes[s]))
        hiddens = _all_reduce_residual(hiddens, partials, [m[2] for m in mods], segments)
        partials = []
        for s, dev in enumerate(DEVICES):
            shard = self.shards[s]
            with _on(dev):
                h = _norm_modulate(shard.norm2, hiddens[s], [mods[s][3], mods[s][4]], segments)
                partials.append(shard.mlp([h]))
        hiddens = _all_reduce_residual(hiddens, partials, [m[5] for m in mods], segments)
        _tp_cache["src"], _tp_cache["replica"] = hiddens[1], hiddens[0]
        return hiddens[1]

    def _forward_sliced(self, hiddens, tembs, ropes, segments):
        mods, partials = [], []
        for s, dev in enumerate(DEVICES):  # kernels are queued on both GPUs before either is waited for
            with _on(dev):
                mods.append(self.shards[s].adaln_proj(tembs[s]))
                partials.append(_sliced_attention(self.shards[s], hiddens[s], mods[s], segments, ropes[s]))
        hiddens = _all_reduce_sliced(hiddens, partials, [m[2] for m in mods], segments)
        partials = []
        for s, dev in enumerate(DEVICES):
            with _on(dev):
                partials.append(_sliced_mlp(self.shards[s], hiddens[s], mods[s], segments))
        hiddens = _all_reduce_sliced(hiddens, partials, [m[5] for m in mods], segments)
        _tp_cache["src"], _tp_cache["replica"] = hiddens[1], hiddens[0]
        return hiddens[1]


def split_transformer(model, progress=None):
    """Every DiT block becomes a TPBlock; the rest of the transformer (embeddings, final layer) moves to GPU 1."""
    for index in range(len(model.blocks)):
        model.blocks[index] = TPBlock(model.blocks[index], index)
        if progress is not None:
            progress(index + 1)
    for name, child in model.named_children():
        if name not in ("blocks", "token_refiner"):
            child.to(MAIN_DEV)
    for name, buffer in list(model.named_buffers(recurse=False)):
        model._buffers[name] = buffer.to(MAIN_DEV)
# <<< section

# ==== Overlapped GPU exchange: the partial sums cross between the T4s while the GPUs keep computing ====
# The two T4s cannot copy to each other directly, so every exchange goes through host RAM: GPU -> pinned RAM -> GPU.
# Sent in one piece after the whole projection, both GPUs wait for it twice per block. Here everything after attention
# runs on row chunks, in a pipeline: while chunk t's output projection is computed, chunk t-1 (whose attention sum has
# arrived) goes through its residual and FFN, and chunk t-2 (whose FFN sum has arrived) finishes the block. The copies
# run on side streams (GPU -> RAM on the sender, RAM -> GPU on the receiver) through a ring of pinned slots, ordered by
# CUDA events, so the transfers overlap the compute instead of stopping it. Same kernels on the same rows.
# >>> section:overlap
OVERLAP = {"on": os.environ.get("PIH3_OVERLAP", "1") == "1", "rows": 4096, "slots": 3, "sample": None, "capture": False}
_LINKS = {}
_LINK_LOCK = threading.Lock()
_SIDE_STREAMS = {}


def _side_streams(s):
    """(send, receive) streams of GPU s, used only from GPU s's own thread."""
    if s not in _SIDE_STREAMS:
        dev = DEVICES[s]
        _SIDE_STREAMS[s] = ((torch.cuda.Stream(device=dev), torch.cuda.Stream(device=dev)) if dev.type == "cuda" else (None, None))
    return _SIDE_STREAMS[s]


class _Link:
    """One direction of one exchange (GPU src -> the other GPU): pinned RAM slots, receive slots on the destination GPU,
    and per slot the events that order them: sent to RAM (d2h), copied onto the destination (h2d), added there (used)."""

    def __init__(self, src, width):
        dst = 1 - src
        nbytes = OVERLAP["rows"] * width * 2
        cuda = DEVICES[dst].type == "cuda"
        self.width = width
        self.host = []
        for _ in range(OVERLAP["slots"]):
            try:
                buf = pinned_empty_exact(nbytes) if cuda else torch.empty(nbytes, dtype=torch.uint8)
            except Exception:
                buf = torch.empty(nbytes, dtype=torch.uint8).pin_memory()
            self.host.append(buf)
        PINNED_BYTES["GPU exchange"] = PINNED_BYTES.get("GPU exchange", 0) + nbytes * OVERLAP["slots"]
        self.dev = [torch.empty(nbytes, dtype=torch.uint8, device=DEVICES[dst]) for _ in range(OVERLAP["slots"])]
        self.d2h = [torch.cuda.Event() if cuda else None for _ in range(OVERLAP["slots"])]
        self.h2d = [torch.cuda.Event() if cuda else None for _ in range(OVERLAP["slots"])]
        self.used = [torch.cuda.Event() if cuda else None for _ in range(OVERLAP["slots"])]


def _link(name, src, width):
    key = (name, src)
    link = _LINKS.get(key)
    if link is None:
        with _LINK_LOCK:  # both threads may ask for it at the same moment: one object
            link = _LINKS.get(key)
            if link is None:
                link = _LINKS[key] = _Link(src, width)
    if link.width != width:
        raise RuntimeError("GPU exchange width changed")
    return link


def _view(buf, rows, width, dtype):
    return buf[:rows * width * 2].view(dtype).view(rows, width)


def _send(s, name, index, chunk):
    """Queue chunk (GPU s's partial sum of some rows) into pinned RAM on GPU s's send stream."""
    link = _link(name, s, chunk.shape[-1])
    k = index % OVERLAP["slots"]
    host = _view(link.host[k], chunk.shape[0], chunk.shape[-1], chunk.dtype)
    send, _ = _side_streams(s)
    if send is None:  # CPU test run
        host.copy_(chunk)
        return
    ready = torch.cuda.Event()
    ready.record(torch.cuda.current_stream(DEVICES[s]))
    send.wait_event(ready)          # the chunk is computed
    send.wait_event(link.h2d[k])    # the slot's previous contents reached the other GPU
    with torch.cuda.stream(send):
        host.copy_(chunk, non_blocking=True)
        link.d2h[k].record(send)
    chunk.record_stream(send)


def _queue_receive(s, name, index, rows, width):
    """After the barrier: queue the other GPU's chunk from pinned RAM onto GPU s, on GPU s's receive stream."""
    link = _link(name, 1 - s, width)
    k = index % OVERLAP["slots"]
    nbytes = rows * width * 2
    _, recv = _side_streams(s)
    if recv is None:
        link.dev[k][:nbytes].copy_(link.host[k][:nbytes])
        return
    recv.wait_event(link.d2h[k])    # the other GPU's chunk is in RAM
    recv.wait_event(link.used[k])   # this GPU added the slot's previous contents
    with torch.cuda.stream(recv):
        link.dev[k][:nbytes].copy_(link.host[k][:nbytes], non_blocking=True)
        link.h2d[k].record(recv)


def _combine(s, name, index, own):
    """own + the other GPU's chunk, in place, on GPU s's compute stream."""
    k = index % OVERLAP["slots"]
    out_link, in_link = _link(name, s, own.shape[-1]), _link(name, 1 - s, own.shape[-1])
    stream = torch.cuda.current_stream(DEVICES[s]) if DEVICES[s].type == "cuda" else None
    if stream is not None:
        stream.wait_event(out_link.d2h[k])  # own chunk has been read out before it is changed
        stream.wait_event(in_link.h2d[k])   # the other GPU's chunk is here
    own.add_(_view(in_link.dev[k], own.shape[0], own.shape[-1], own.dtype))
    if stream is not None:
        in_link.used[k].record(stream)
    return own


def _no_projection(attention_list, seq_len, residual, gate, segments):
    return _take(attention_list).reshape(seq_len, -1)


def _attention_rows(shard, hidden, mod, segments, rope, sliced):
    """GPU s's attention heads for all rows, before the output projection."""
    if sliced:
        return _sliced_attention_rows(shard, hidden, mod, segments, rope)
    attn = shard.attn
    attn.__dict__["_out_projection"] = _no_projection  # H3's own attention path, stopped before out_proj
    try:
        return attn([_norm_modulate(shard.norm1, hidden, [mod[0], mod[1]], segments)], rope=rope)
    finally:
        attn.__dict__.pop("_out_projection", None)


class _RowView:
    """Rows a..b of an Int8Shard (views, no copy): the Q, K or V rows of a group of heads. Runs Int8Shard's forward."""

    def __init__(self, shard, a, b):
        self.qdata, self.qscale, self.group = shard.qdata[a:b], shard.qscale[a:b], shard.group
        self.out_features, self.in_features = b - a, shard.in_features

    __call__ = Int8Shard.forward


_ROW_VIEWS = {}


def _qkv_views(qkv_proj, first_head, heads, dim, inner):
    key = (id(qkv_proj), first_head, heads)
    if key not in _ROW_VIEWS:
        a, b = first_head * dim, (first_head + heads) * dim
        _ROW_VIEWS[key] = tuple(_RowView(qkv_proj, part * inner + a, part * inner + b) for part in range(3))
    return _ROW_VIEWS[key]


# Long-video attention by groups of heads: only one group's Q / K / V (and the copies the attention kernel makes of them)
# exist at full length at a time, next to the residual stream and the attention output. Heads are independent, so the
# result is the same as with all heads at once.
LONG_VIDEO["head_groups"] = 4


def _sliced_attention_rows(shard, hidden, mod, segments, rope):
    attn = shard.attn
    rows, heads, dim = hidden.shape[0], attn.heads, attn.head_dim
    inner = heads * dim
    if isinstance(rope, tuple):
        rope = rope[0]
    per = -(-heads // max(1, min(heads, LONG_VIDEO["head_groups"])))
    out = torch.empty(rows, inner, device=hidden.device, dtype=torch.float16)
    for first in range(0, heads, per):
        count = min(per, heads - first)
        views = _qkv_views(attn.qkv_proj, first, count, dim, inner)
        q, k, v = (torch.empty(1, rows, count, dim, device=hidden.device, dtype=torch.float16) for _ in range(3))
        for a, b in _row_chunks(rows):
            h = _norm_modulate(shard.norm1, hidden[a:b], [mod[0], mod[1]], _slice_segments(segments, a, b))
            query, key, value = (view(h).reshape(1, b - a, count, dim) for view in views)
            del h
            query, key = attn.q_norm(query), attn.k_norm(key)
            if rope is not None:
                _apply_rope(query, key, rope[:, a:b])
            q[:, a:b].copy_(query)
            k[:, a:b].copy_(key)
            v[:, a:b].copy_(value)
            del query, key, value
        qkv_list = [q, k, v]
        del q, k, v
        out[:, first * dim:(first + count) * dim].copy_(_h3_transformer.pay_attention(qkv_list, recycle_q=True)[0].reshape(rows, count * dim))
    return out


def _half_block_overlap(s, block, hidden, temb, rope, segments, shared, sliced):
    """GPU s's half of a DiT block with the exchanges pipelined over row chunks (see above)."""
    shard = block.shards[s]
    mod = shard.adaln_proj(temb)
    rows_out = _attention_rows(shard, hidden, mod, segments, rope, sliced)
    total, width = hidden.shape[0], hidden.shape[-1]
    chunks = [(a, min(total, a + OVERLAP["rows"])) for a in range(0, total, OVERLAP["rows"])]
    n = len(chunks)
    own_attn, own_mlp = {}, {}
    for t in range(n + 2):
        if t < n:  # chunk t: output projection, sent
            a, b = chunks[t]
            own_attn[t] = shard.attn.out_proj(rows_out[a:b].to(hidden.dtype) if sliced else rows_out[a:b])  # as each mode's plain path
            _send(s, "attn", t, own_attn[t])
            if t == n - 1:
                rows_out = None
        j = t - 1
        if 0 <= j < n:  # chunk t-1: attention sum + residual, FFN, sent
            a, b = chunks[j]
            seg = _slice_segments(segments, a, b)
            _gated_residual([hidden[a:b]], [mod[2]], [_combine(s, "attn", j, own_attn.pop(j))], seg)
            own_mlp[j] = shard.mlp([_norm_modulate(shard.norm2, hidden[a:b], [mod[3], mod[4]], seg)])
            _send(s, "mlp", j, own_mlp[j])
        z = t - 2
        if 0 <= z < n:  # chunk t-2: FFN sum + residual
            a, b = chunks[z]
            _gated_residual([hidden[a:b]], [mod[5]], [_combine(s, "mlp", z, own_mlp.pop(z))], _slice_segments(segments, a, b))
        _TP_BARRIER.wait()  # both GPUs queued this round's sends
        if t < n:
            _queue_receive(s, "attn", t, chunks[t][1] - chunks[t][0], width)
        if 0 <= j < n:
            _queue_receive(s, "mlp", j, chunks[j][1] - chunks[j][0], width)
    return hidden


_half_block_exchange = _half_block


def _half_block(s, block, hidden, temb, rope, segments, shared, sliced):
    if OVERLAP["on"] and hidden.element_size() == 2:
        return _half_block_overlap(s, block, hidden, temb, rope, segments, shared, sliced)
    return _half_block_exchange(s, block, hidden, temb, rope, segments, shared, sliced)
# <<< section



NUM_BLOCKS = len(transformer.blocks)
BAR.start("split", "Loading: splitting the DiT", NUM_BLOCKS, "blocks")
split_transformer(transformer, BAR.update)
BAR.finish()
gc.collect()

# ==== Exact-size pinned RAM for the token refiner ====
# The 1.4 GiB token refiner runs once per prompt: keep it in page-locked RAM (cudaHostRegister: exactly the bytes asked
# for, unlike PyTorch's power-of-two pinned blocks) and swap it onto GPU 1 only for that.
_HOST_REGISTERED = []  # registered RAM stays alive (and pinned) for the whole session
PINNED_BYTES = {}


def _dont_fork(ptr, nbytes):
    """madvise(MADV_DONTFORK): a fork() must not duplicate pinned RAM (Linux copies pinned pages into the child at once)."""
    try:
        import ctypes
        page = os.sysconf("SC_PAGE_SIZE")
        start = -(-ptr // page) * page
        end = (ptr + nbytes) // page * page
        if end > start:
            ctypes.CDLL("libc.so.6").madvise(ctypes.c_void_p(start), ctypes.c_size_t(end - start), 10)  # MADV_DONTFORK
    except Exception:
        pass


def pinned_empty_exact(nbytes):
    t = torch.empty(nbytes, dtype=torch.uint8)
    torch.cuda.check_error(torch.cuda.cudart().cudaHostRegister(t.data_ptr(), nbytes, 0))
    _dont_fork(t.data_ptr(), nbytes)
    _HOST_REGISTERED.append(t)
    return t


_refiner = transformer.token_refiner
_ref_list = list(_refiner.parameters())
_ref_offsets, _ref_total = [], 0
for _p in _ref_list:
    _ref_offsets.append(_ref_total)
    _ref_total += -(-_p.numel() * _p.element_size() // 64) * 64
try:
    _ref_buf = pinned_empty_exact(_ref_total)
    _refiner_params = []
    for _p, _off in zip(_ref_list, _ref_offsets):
        _cpu = _ref_buf[_off:_off + _p.numel() * _p.element_size()].view(_p.dtype).view(_p.shape)
        _cpu.copy_(_p.data)
        _refiner_params.append((_p, _cpu))
except Exception as e:  # no cudaHostRegister: PyTorch's (rounded) pinned blocks
    print(f"⚠️ Exact pinning unavailable ({e}); using PyTorch pinned memory")
    _refiner_params = [(p, p.data.to("cpu").pin_memory()) for p in _ref_list]
PINNED_BYTES["token refiner"] = _ref_total
for _p, _cpu in _refiner_params:
    _p.data = _cpu
_orig_preprocess_text = transformer.preprocess_text_embeds


def _preprocess_text_embeds(text_states):
    if text_states.shape[-1] == transformer.hidden_size:
        return text_states
    for p, cpu in _refiner_params:
        p.data = cpu.to(MAIN_DEV, non_blocking=True)
    try:
        return _orig_preprocess_text(text_states.to(MAIN_DEV))
    finally:
        for p, cpu in _refiner_params:
            p.data = cpu



transformer.preprocess_text_embeds = _preprocess_text_embeds

# ==== pi-H3: the 2-step and 4-step adapters ====
# Each adapter changes four things, all kept in Wan2GP's layer names:
# * blocks 0-45: rank-64 LoRAs on the attention output, FFN-up and FFN-down projections. They run live in FP16 on top of
#   the INT8 weights (never merged into them: re-rounding INT8 weights would erase the small deltas).
# * every block's AdaLN curve projection and the output head (FP32, small): replaced whole.
# * blocks 46-49: attention output and FFN replaced whole, as INT8 ConvRot weights quantized once from the adapter.
# * the DWD pixel head (below).
# Both adapters have exactly the same tensor shapes. One is on the GPUs at a time; both stay in exact-size pinned RAM and
# switching copies the other one into the same GPU buffers (~2.4 GB, about a second).
# The VAE-decoder modes use pi-H3's 2 and 4-step adapters made for a regular decode: the same LoRAs, AdaLN curves and
# output head as the DWD adapter of the same step count (stored once), plus LoRAs on blocks 46-49, which keep the base model's own weights.
# >>> section:pi
from safetensors import safe_open

PI_STEPS = (2, 4)
PI_MODES = {"dwd2": 2, "dwd4": 4, "vae2": 2, "vae4": 4}  # mode -> steps; the vae modes decode the latent with the VAE decoder
VAE_MODES = ("vae2", "vae4")
PI_REPLACED_BLOCKS = (46, 47, 48, 49)
PI_MAX_RAW_T = {2: 0.93, 4: 0.96}  # first time of the trained schedules, before the flow shift
PI_FINAL_STEP_SCALE = 0.6          # the last step is this fraction of the others
PI_SHIFT = 12.0


def pi_sigmas(steps, shift=PI_SHIFT):
    """The video sigmas pi-H3 was trained on: equal steps in unshifted time from max_raw_t, a shorter last one, then the
    flow shift (2 steps: 0.99377, 0.86534). The audio sigmas follow from them with H3's audio shift."""
    first = PI_MAX_RAW_T[int(steps)]
    raw = torch.linspace(first, first * PI_FINAL_STEP_SCALE / (int(steps) - 1 + PI_FINAL_STEP_SCALE), int(steps), dtype=torch.float64)
    return (shift * raw / (1.0 + (shift - 1.0) * raw)).tolist()


class LoraRuntime:
    """The rank-64 LoRAs on the tensor-parallel halves. Row-split layers (FFN-up) take the matching rows of B,
    column-split layers (attention output, FFN-down) the matching columns of A (their partial outputs are summed across
    the GPUs anyway). The factors stay in FP16 on the GPUs and the LoRA math runs on the FP16 tensor cores."""

    ROWS = 8192

    def __init__(self, model):
        self.targets = {}  # name -> [(module, device, kind, index)]
        self.ids = {}  # name -> {id(module) of its GPU halves}
        for i, block in enumerate(model.blocks):
            for short, attr, kind in (("out", "attn.out_proj", "cols"), ("fc1", "mlp.fc1", "rows"), ("fc2", "mlp.fc2", "cols")):
                self.targets[f"blocks.{i}.{attr}"] = [(functools.reduce(getattr, attr.split("."), block.shards[s]), DEVICES[s],
                                                       kind, block.splits[s][short]) for s in range(len(DEVICES))]
        self.adapters = {}  # id(module) -> (A, [(B, out_start, out_end, rank_start, rank_end)], scale)
        self.handles = []
        self.strength = 0.0
        self.off = set()  # id(module) of LoRAs the active adapter does not have (blocks 46-49 of the DWD adapters)

    def split(self, groups):
        """{name: (A, B)} -> {(name, gpu index): (A half, B half)} as FP16 CPU tensors."""
        out = {}
        for name, (a, b) in groups.items():
            for s, (module, dev, kind, index) in enumerate(self.targets.get(name, ())):
                if kind == "rows":
                    out[(name, s)] = (a.to(torch.float16).contiguous(), b.index_select(0, index).to(torch.float16).contiguous())
                else:
                    out[(name, s)] = (a[:, index].to(torch.float16).contiguous(), b.to(torch.float16).contiguous())
        return out

    def attach(self, parts):
        """GPU buffers for the factors (filled by the adapter switch) and the forward hooks."""
        gpu = {}
        for (name, s), (a, b) in parts.items():
            module, dev, _, _ = self.targets[name][s]
            a_gpu = torch.empty_like(a, device=dev)
            b_gpu = torch.empty_like(b, device=dev)
            self.adapters[id(module)] = (a_gpu, [(b_gpu, 0, b.shape[0], 0, a.shape[0])], 1.0)
            self.ids[name] = self.ids.get(name, set()) | {id(module)}
            self.handles.append(module.register_forward_hook(self._hook))
            gpu[f"lora.{name}.{s}.A"], gpu[f"lora.{name}.{s}.B"] = a_gpu, b_gpu
        return gpu

    def _hook(self, module, args, output):
        if self.strength == 0.0 or id(module) in self.off:
            return output
        a16, outs, alpha = self.adapters[id(module)]
        s = self.strength * alpha
        x = args[0]
        x2 = x.reshape(-1, x.shape[-1])
        out2 = output.view(-1, output.shape[-1])
        # The input is scaled by a power of two before the FP16 cast and the result scaled back, so large activations
        # (FFN-down input) cannot overflow; FP16's 11-bit mantissa is plenty for a correction term.
        pre = 1.0 / 512 if x2.shape[-1] > 8192 else 1.0 / 64
        for start in range(0, x2.shape[0], self.ROWS):
            end = min(x2.shape[0], start + self.ROWS)
            h = (x2[start:end] * pre).to(torch.float16) @ a16.t()
            for b16, o0, o1, r0, r1 in outs:
                out2[start:end, o0:o1].add_((h[:, r0:r1] @ b16.t()).to(out2.dtype), alpha=s / pre)
            del h
        return output


# ---- DWD pixel head: the last 8 layers of Sol-H3-VAE's decoder on the DiT's features ----
# At the last step the video rows after the last DiT block go through a weightless LayerNorm modulated from the step's
# time (the head's own AdaLN), then a linear expansion turns every DiT token (2 x 2 latents) into 2 x 2 Sol tokens of
# 2048 channels, one per latent (16 x 16 pixels). Eight Sol layers refine them: window attention over 5 latents x
# 16 x 16 tokens (odd layers shift the windows by 2 latents and 8 tokens) plus 5 global tokens per native 5-latent
# block, then a SwiGLU FFN. The first latent of each block projects to 1 frame, the others to 4 frames each (17 frames
# per block, as H3's VAE). The layers here stream over the native blocks: only two blocks' Q / K / V exist at a time,
# so the memory stays small at any length. Same arithmetic as the reference decoder (LakonLab's SolH3DWDHead).
SOL_DIM, SOL_HEADS, SOL_HEAD_DIM, SOL_FFN = 2048, 32, 64, 8192
SOL_WINDOW, SOL_BLOCK, SOL_T_SHIFT, SOL_TIME_NORM = 16, 5, 2, 5
SOL_GLOBALS, SOL_LAYERS, SOL_FIRST_LAYER = 5, 8, 16  # 4 register tokens + a class token; layers 16-23 of the 24-layer decoder
SOL_PATCH, SOL_PATCH_T, SOL_ROPE_PAIRS = 16, 4, 24
SOL_COLOR_MEAN, SOL_COLOR_STD = (-0.03, -0.088, -0.188), (0.458, 0.448, 0.45)  # Sol's ImageNet-normalized RGB -> [-1, 1]
SOL_ROWS = 8192           # rows per GEMM chunk
SOL_WINDOW_GROUP = {"n": 8}  # spatial windows per attention call (set from free VRAM before each decode)


class QLinear(torch.nn.Module):
    """An INT8 ConvRot linear with a bias (the pixel head's layers): Kitchen's INT8 GEMM, or the reference path."""

    def __init__(self, out_features, in_features, device, group=256):
        super().__init__()
        self.register_buffer("qdata", torch.empty(out_features, in_features, dtype=torch.int8, device=device), persistent=False)
        self.register_buffer("qscale", torch.empty(out_features, dtype=torch.float32, device=device), persistent=False)
        self.register_buffer("bias", torch.empty(out_features, dtype=torch.bfloat16, device=device), persistent=False)
        self.group, self.out_features, self.in_features = group, out_features, in_features

    def forward(self, x):
        x2 = x.reshape(-1, self.in_features).contiguous()
        bias = self.bias.to(x2.dtype)
        if int8_backend.kitchen_enabled() and x2.is_cuda:
            out = int8_backend._linear_impl(x2, self.qdata, self.qscale, bias, True)
        else:
            weight = self.qdata.to(x2.dtype) * self.qscale.to(x2.dtype)[:, None]
            out = torch.nn.functional.linear(_rotate_activation(x2, self.group), weight, bias)
        return out.reshape(*x.shape[:-1], self.out_features)


def _rms(x, weight=None, eps=1e-5):
    """RMSNorm in FP32 (x already FP32)."""
    x = x * torch.rsqrt(x.square().mean(-1, keepdim=True).add_(eps))
    return x if weight is None else x.mul_(weight.float())


def _sol_rotate(x, cos, sin):
    """Sol's rotary embedding on [n, heads, 64] in place: the first 48 channels as 24 split-half pairs, rounded like the
    reference's out-of-place version in x's dtype."""
    cos, sin = cos.to(x.dtype)[:, None], sin.to(x.dtype)[:, None]
    first, second = x[..., :SOL_ROPE_PAIRS], x[..., SOL_ROPE_PAIRS:2 * SOL_ROPE_PAIRS]
    first_sin, second_sin = first * sin, second * sin
    first.mul_(cos).sub_(second_sin)
    second.mul_(cos).add_(first_sin)


def _sol_layer_buffers(module, device):
    """A Sol layer's tensors on `device`: the four INT8 linears and its norm / scale vectors (filled later)."""
    D = SOL_DIM
    module.qkv, module.out = QLinear(3 * D, D, device), QLinear(D, D, device)
    module.w1, module.w2 = QLinear(2 * SOL_FFN, D, device), QLinear(D, SOL_FFN, device)
    for name in ("norm1", "norm2", "scale1", "scale2"):
        module.register_buffer(name, torch.empty(D, dtype=torch.bfloat16, device=device), persistent=False)
    return module


def _sol_pixels(px, rounding):
    """Sol's ImageNet-normalized RGB [3, frames, H, W] -> [-1, 1], then uint8 as Wan2GP converts its decoded videos
    ("truncate": FP16, truncated; "round": rounded) or kept as floats (no rounding)."""
    mean = torch.tensor(SOL_COLOR_MEAN, device=px.device).view(3, 1, 1, 1)
    std = torch.tensor(SOL_COLOR_STD, device=px.device).view(3, 1, 1, 1)
    px = px.mul_(std).add_(mean).clamp_(-1.0, 1.0)
    if rounding == "round":
        return px.add_(1.0).mul_(127.5).round_().clamp_(0, 255).to(torch.uint8)
    if rounding is None:
        return px
    return px.half().add_(1.0).mul_(127.5).clamp_(0, 255).to(torch.uint8)


class SolLayers(torch.nn.Module):
    """The layers of Sol-H3-VAE's decoder over a whole clip: window attention over 5 latents x 16 x 16 tokens (odd
    layers shift the windows by 2 latents and 8 tokens) plus 5 global tokens per native 5-latent block, then a SwiGLU
    FFN. The layers stream over the native blocks: only two blocks' Q / K / V exist at a time, so the memory stays small
    at any length. Same arithmetic as the reference decoder."""

    def _rope(self, lat_lo, lat_hi, Hs, Ws, s, st, dev):
        inv = 1.0 / (100.0 ** torch.arange(0, 1, 6 / (2 * SOL_ROPE_PAIRS), dtype=torch.float32, device=dev))
        lt = ((torch.arange(lat_lo, lat_hi, device=dev) + st) % SOL_BLOCK).float()
        ly = ((torch.arange(Hs, device=dev) + s) % SOL_WINDOW).float()
        lx = ((torch.arange(Ws, device=dev) + s) % SOL_WINDOW).float()
        grid = torch.stack(torch.meshgrid((lt + 0.5) / SOL_TIME_NORM * 2 - 1, (ly + 0.5) / SOL_WINDOW * 2 - 1,
                                          (lx + 0.5) / SOL_WINDOW * 2 - 1, indexing="ij"), -1).reshape(-1, 3)
        angles = (2.0 * math.pi * grid[:, :, None] * inv[None, None, :]).flatten(1)  # [n, 24]: time, row, column
        return angles.cos(), angles.sin()

    def _qkv(self, layer, rows, rope):
        """FP16 q, k, v [n, heads, 64] of BF16 rows [n, 2048] (RMS-normed, q / k normed per head, roped)."""
        n = rows.shape[0]
        q, k, v = (torch.empty(n, SOL_HEADS, SOL_HEAD_DIM, dtype=torch.float16, device=rows.device) for _ in range(3))
        for a in range(0, n, SOL_ROWS):
            b = min(n, a + SOL_ROWS)
            qkv = layer.qkv(_rms(rows[a:b].float(), layer.norm1).to(torch.bfloat16)).view(b - a, SOL_HEADS, 3, SOL_HEAD_DIM)
            qq = _rms(qkv[:, :, 0].float()).to(torch.bfloat16)
            kk = _rms(qkv[:, :, 1].float()).to(torch.bfloat16)
            if rope is not None:
                _sol_rotate(qq, rope[0][a:b], rope[1][a:b])
                _sol_rotate(kk, rope[0][a:b], rope[1][a:b])
            q[a:b], k[a:b], v[a:b] = qq, kk, qkv[:, :, 2]
            del qkv, qq, kk
        return q, k, v

    def _ffn_(self, layer, rows):
        """rows += FFN(rows) * scale2, in place, by chunks."""
        for a in range(0, rows.shape[0], SOL_ROWS):
            b = min(rows.shape[0], a + SOL_ROWS)
            h = layer.w1(_rms(rows[a:b].float(), layer.norm2).to(torch.bfloat16))
            half = h.shape[-1] // 2
            gate = torch.nn.functional.silu(h[:, :half]).mul_(h[:, half:])
            rows[a:b].add_(layer.w2(gate) * layer.scale2)
            del h, gate

    def _window_attention(self, q, k, v, gk, gv, gvalid, lat_lo, lat_hi, slot0, Hs, Ws, s):
        """Shifted-window attention of one temporal window: q / k / v [n_lat, Hs, Ws, heads, 64] of its tokens, gk / gv
        [2 * S, heads, 64] the global tokens of the native blocks it overlaps. Returns [n_lat, Hs, Ws, 2048]."""
        n_lat = lat_hi - lat_lo
        Hp, Wp = -(-(Hs + s) // SOL_WINDOW) * SOL_WINDOW, -(-(Ws + s) // SOL_WINDOW) * SOL_WINDOW
        nh, nw = Hp // SOL_WINDOW, Wp // SOL_WINDOW
        L = SOL_BLOCK * SOL_WINDOW * SOL_WINDOW
        keys = L + gk.shape[0]
        keys_padded = -(-keys // 16) * 16  # an aligned key count for the memory-efficient kernel
        dev, dt = q.device, q.dtype

        def windows(t):
            padded = t.new_zeros(SOL_BLOCK, Hp, Wp, SOL_HEADS, SOL_HEAD_DIM)
            padded[slot0:slot0 + n_lat, s:s + Hs, s:s + Ws] = t
            return padded.view(SOL_BLOCK, nh, SOL_WINDOW, nw, SOL_WINDOW, SOL_HEADS, SOL_HEAD_DIM).permute(
                1, 3, 0, 2, 4, 5, 6).reshape(nh * nw, L, SOL_HEADS, SOL_HEAD_DIM)

        valid = torch.zeros(SOL_BLOCK, Hp, Wp, dtype=torch.bool, device=dev)
        valid[slot0:slot0 + n_lat, s:s + Hs, s:s + Ws] = True
        valid = valid.view(SOL_BLOCK, nh, SOL_WINDOW, nw, SOL_WINDOW).permute(1, 3, 0, 2, 4).reshape(nh * nw, L)
        qw, kw, vw = windows(q), windows(k), windows(v)
        out = torch.empty(nh * nw, L, SOL_HEADS, SOL_HEAD_DIM, dtype=dt, device=dev)
        for a in range(0, nh * nw, SOL_WINDOW_GROUP["n"]):
            b = min(nh * nw, a + SOL_WINDOW_GROUP["n"])
            n = b - a
            kk = torch.zeros(n, keys_padded, SOL_HEADS, SOL_HEAD_DIM, dtype=dt, device=dev)
            vv = torch.zeros_like(kk)
            kk[:, :L], vv[:, :L] = kw[a:b], vw[a:b]
            kk[:, L:keys], vv[:, L:keys] = gk, gv
            mask = torch.zeros(n, keys_padded, dtype=torch.bool, device=dev)
            mask[:, :L] = valid[a:b]
            mask[:, L:keys] = gvalid
            o = torch.nn.functional.scaled_dot_product_attention(
                qw[a:b].transpose(1, 2), kk.transpose(1, 2), vv.transpose(1, 2), attn_mask=mask[:, None, None, :])
            out[a:b] = o.transpose(1, 2)
            del kk, vv, mask, o
        del qw, kw, vw
        out = out.view(nh, nw, SOL_BLOCK, SOL_WINDOW, SOL_WINDOW, SOL_DIM).permute(2, 0, 3, 1, 4, 5).reshape(SOL_BLOCK, Hp, Wp, SOL_DIM)
        return out[slot0:slot0 + n_lat, s:s + Hs, s:s + Ws]

    def _layer(self, layer, shifted, x, g, T, t_off=0, windows=None):
        """One Sol layer on the tokens x [latents, Hs, Ws, 2048] (latent t at x[t - t_off], T latents in the clip) and the
        global tokens g [blocks, 5, 2048], BF16, both in place. windows: the temporal windows (first, last + 1) to run,
        all by default; only their tokens change, and only the global tokens of the native blocks they reach are valid
        afterwards."""
        _, Hs, Ws, _ = x.shape
        nb = g.shape[0]
        s, st = (SOL_WINDOW // 2, SOL_T_SHIFT) if shifted else (0, 0)
        gq, gk, gv = self._qkv(layer, g.reshape(-1, SOL_DIM), None)  # the global tokens: no rotary embedding
        gq, gk, gv = (t.view(nb, SOL_GLOBALS, SOL_HEADS, SOL_HEAD_DIM) for t in (gq, gk, gv))
        g_out = torch.zeros(nb, SOL_GLOBALS, SOL_DIM, dtype=x.dtype, device=x.device)
        ring = {}

        def ensure(blk):
            """Q / K / V of native block blk's tokens from the layer's input, and its global tokens' attention."""
            if blk in ring:
                return
            a, b = blk * SOL_BLOCK, min(T, (blk + 1) * SOL_BLOCK)
            q, k, v = self._qkv(layer, x[a - t_off:b - t_off].reshape(-1, SOL_DIM), self._rope(a, b, Hs, Ws, s, st, x.device))
            ring[blk] = tuple(t.view(b - a, Hs, Ws, SOL_HEADS, SOL_HEAD_DIM) for t in (q, k, v))
            # global tokens of the block attend to every token of the block and to the block's global tokens
            keys = torch.cat((k, gk[blk]))[None].transpose(1, 2)
            values = torch.cat((v, gv[blk]))[None].transpose(1, 2)
            o = torch.nn.functional.scaled_dot_product_attention(gq[blk][None].transpose(1, 2), keys, values)
            g_out[blk] = o.transpose(1, 2).reshape(SOL_GLOBALS, SOL_DIM)
            del keys, values, o
            for old in [key for key in ring if key < blk - 1]:
                del ring[old]

        num_windows = -(-(T + st) // SOL_BLOCK)
        first, last = windows if windows is not None else (0, num_windows)
        for j in range(first, min(last, num_windows)):
            t0 = j * SOL_BLOCK - st
            lat_lo, lat_hi = max(t0, 0), min(t0 + SOL_BLOCK, T)
            if lat_hi <= lat_lo:
                continue
            b0, b1 = lat_lo // SOL_BLOCK, (lat_hi - 1) // SOL_BLOCK
            for blk in range(b0, b1 + 1):
                ensure(blk)
            q, k, v = (torch.cat([ring[blk][i][max(lat_lo, blk * SOL_BLOCK) - blk * SOL_BLOCK:
                                               min(lat_hi, (blk + 1) * SOL_BLOCK) - blk * SOL_BLOCK]
                                  for blk in range(b0, b1 + 1)]) for i in range(3))
            gkw = torch.cat((gk[b0], gk[b1]))
            gvw = torch.cat((gv[b0], gv[b1]))
            gvalid = torch.tensor([True] * SOL_GLOBALS + [b1 != b0] * SOL_GLOBALS, device=x.device)
            o = self._window_attention(q, k, v, gkw, gvw, gvalid, lat_lo, lat_hi, lat_lo - t0, Hs, Ws, s)
            del q, k, v
            rows = x[lat_lo - t_off:lat_hi - t_off].view(-1, SOL_DIM)
            for a in range(0, rows.shape[0], SOL_ROWS):  # attention residual, then the FFN of these finished tokens
                b = min(rows.shape[0], a + SOL_ROWS)
                rows[a:b].add_(layer.out(o.reshape(-1, SOL_DIM)[a:b].to(torch.bfloat16)) * layer.scale1)
            del o
            self._ffn_(layer, rows)
        ring.clear()
        g.view(-1, SOL_DIM).add_(layer.out(g_out.view(-1, SOL_DIM)) * layer.scale1)
        self._ffn_(layer, g.view(-1, SOL_DIM))


class DWDHead(SolLayers):
    def __init__(self, device, hidden=5376):
        super().__init__()
        D = SOL_DIM

        def buf(module, name, shape, dtype):
            module.register_buffer(name, torch.empty(shape, dtype=dtype, device=device), persistent=False)

        buf(self, "time_table", (1025, 8), torch.float32)
        buf(self, "ada_w", (2 * hidden, 8), torch.float32)
        buf(self, "ada_b", (2 * hidden,), torch.float32)
        buf(self, "fe_w", (4 * D, hidden), torch.float16)
        buf(self, "fe_b", (4 * D,), torch.float16)
        buf(self, "registers", (SOL_GLOBALS - 1, D), torch.bfloat16)
        buf(self, "norm_out_w", (D,), torch.float32)
        buf(self, "norm_out_b", (D,), torch.float32)
        buf(self, "proj_w", (3 * SOL_PATCH_T * SOL_PATCH * SOL_PATCH, D), torch.float32)
        buf(self, "proj_b", (3 * SOL_PATCH_T * SOL_PATCH * SOL_PATCH,), torch.float32)
        buf(self, "first_w", (3 * SOL_PATCH * SOL_PATCH, D), torch.float32)
        buf(self, "first_b", (3 * SOL_PATCH * SOL_PATCH,), torch.float32)
        self.layers = torch.nn.ModuleList(_sol_layer_buffers(torch.nn.Module(), device) for _ in range(SOL_LAYERS))
        self.device = device

    # The DWD module file's tensor names (Wan2GP layout of the adapter's dwd_norm_out / dwd_pixel_refiner)
    def source_names(self):
        names = {"time_table": "dwd_head.time_table", "ada_w": "dwd_head.adaln_proj.linear.weight",
                 "ada_b": "dwd_head.adaln_proj.linear.bias", "fe_w": "dwd_head.refiner.feature_expand.weight",
                 "fe_b": "dwd_head.refiner.feature_expand.bias", "registers": "dwd_head.refiner.register_tokens",
                 "norm_out_w": "dwd_head.refiner.norm_out.weight", "norm_out_b": "dwd_head.refiner.norm_out.bias",
                 "proj_w": "dwd_head.refiner.proj_out.weight", "proj_b": "dwd_head.refiner.proj_out.bias",
                 "first_w": "dwd_head.refiner.first_frame_proj_out.weight", "first_b": "dwd_head.refiner.first_frame_proj_out.bias"}
        for i in range(SOL_LAYERS):
            src = f"dwd_head.refiner.transformer_blocks.{i}"
            for mine, theirs in (("qkv", "attn.to_qkv"), ("out", "attn.to_out"), ("w1", "ff.w1"), ("w2", "ff.w2")):
                names[f"layers.{i}.{mine}.qdata"] = f"{src}.{theirs}.weight"
                names[f"layers.{i}.{mine}.qscale"] = f"{src}.{theirs}.weight_scale"
                names[f"layers.{i}.{mine}.bias"] = f"{src}.{theirs}.bias"
            for name in ("norm1", "norm2"):
                names[f"layers.{i}.{name}"] = f"{src}.{name}.weight"
            for name in ("scale1", "scale2"):
                names[f"layers.{i}.{name}"] = f"{src}.{name}"
        return names

    @torch.no_grad()
    def features(self, hidden, t, out):
        """The head's input: hidden (the video rows after the last DiT block, borrowed) through a weightless LayerNorm
        modulated by (scale, shift) = linear(SiLU(time curve at t)), written to `out` (FP16)."""
        table = self.time_table
        position = min(max(float(t), 0.0), 1.0) * (table.shape[0] - 1)
        lower = min(int(math.floor(position)), table.shape[0] - 2)
        temb = torch.lerp(table[lower], table[lower + 1], position - lower)
        scale, shift = torch.nn.functional.linear(torch.nn.functional.silu(temb), self.ada_w, self.ada_b).chunk(2)
        scale = scale.add_(1.0).to(hidden.device)
        shift = shift.to(hidden.device)
        for start in range(0, hidden.shape[0], 4096):
            stop = min(start + 4096, hidden.shape[0])
            h = torch.nn.functional.layer_norm(hidden[start:stop].float(), (hidden.shape[-1],), eps=1e-5).mul_(scale).add_(shift)
            out[start:stop].copy_(h, non_blocking=True)
        return out

    @torch.no_grad()
    def decode(self, features_list, grid, progress=None):
        """FP16 features [t * h * w, DiT width] on the (t, h, w) DiT token grid (handed over in a list, released once
        expanded) -> uint8 video [3, frames, 32 h, 32 w] in RAM."""
        features = features_list.pop()
        t, h, w = grid
        dev = self.device
        T, Hs, Ws = t, 2 * h, 2 * w
        nb = -(-T // SOL_BLOCK)
        x = torch.empty(T, Hs, Ws, SOL_DIM, dtype=torch.bfloat16, device=dev)
        per_frame = h * w
        for f in range(T):  # every DiT token -> its 2 x 2 Sol tokens
            rows = features[f * per_frame:(f + 1) * per_frame].to(dev, torch.float16, non_blocking=True)
            expanded = torch.nn.functional.linear(rows, self.fe_w, self.fe_b).view(h, w, 2, 2, SOL_DIM)
            x[f] = expanded.permute(0, 2, 1, 3, 4).reshape(Hs, Ws, SOL_DIM)
            del rows, expanded
        del features
        if dev.type == "cuda":  # windows per attention call: room for the scores even if SDPA takes its plain path
            free = torch.cuda.mem_get_info(dev)[0] - (2 << 30)
            SOL_WINDOW_GROUP["n"] = int(min(16, max(2, free // (400 * 2**20))))
        g = torch.cat((self.registers, self.registers.new_zeros(1, SOL_DIM)))[None].repeat(nb, 1, 1)
        for index in range(SOL_LAYERS):
            self._layer(self.layers[index], (SOL_FIRST_LAYER + index) % 2 == 1, x, g, T)
            if progress is not None:
                progress(index + 1, SOL_LAYERS + 1)
        del g
        frames = sum(1 + SOL_PATCH_T * (min(T, (blk + 1) * SOL_BLOCK) - blk * SOL_BLOCK - 1) for blk in range(nb))
        P = SOL_PATCH
        video = torch.empty(3, frames, Hs * P, Ws * P, dtype=torch.uint8)
        frame = 0
        for lat in range(T):
            y = torch.nn.functional.layer_norm(x[lat].reshape(-1, SOL_DIM).float(), (SOL_DIM,), self.norm_out_w, self.norm_out_b, 1e-5)
            if lat % SOL_BLOCK == 0:  # the first latent of a native block: one frame
                px = torch.nn.functional.linear(y, self.first_w, self.first_b).view(Hs, Ws, 3, 1, P, P)
            else:
                px = torch.nn.functional.linear(y, self.proj_w, self.proj_b).view(Hs, Ws, 3, SOL_PATCH_T, P, P)
            n = px.shape[3]
            video[:, frame:frame + n] = _sol_pixels(px.permute(2, 3, 0, 4, 1, 5).reshape(3, n, Hs * P, Ws * P), "truncate").cpu()
            frame += n
            del y, px
        del x
        if progress is not None:
            progress(SOL_LAYERS + 1, SOL_LAYERS + 1)
        return video


# ---- The VAE decoder: Sol-H3-VAE (NVIDIA), distilled from H3's 36-layer VAE decoder ----
# 24 Sol layers on the latents themselves (one token per latent: 16 x 16 pixels x 4 frames, the first latent of each
# native block keeping its last frame), whole frames with window attention instead of H3's overlapping 256-pixel tiles.
# Both T4s decode at once: GPU 0 takes the first native blocks, GPU 1 the rest. A window only reaches its own block or,
# shifted, the last two latents of the previous one, so before each layer the GPUs only swap the rows at the border
# (5 latents one way, 2 the other, through pinned RAM): each GPU then computes exactly what a single GPU would.
# The INT8 weights stay in pinned RAM; each GPU copies one layer at a time (67 MB) into a slot of its own.
SOL_SPLIT = {"on": True}


class SolVAEDecoder(SolLayers):
    def __init__(self, filename):
        super().__init__()
        tensors, self.small = {}, {}
        with safe_open(filename, framework="pt", device="cpu") as f:
            keys = set(f.keys())
            count = sum(1 for k in keys if k.startswith("decoder.transformer_blocks.") and k.endswith(".attn.to_qkv.weight"))
            for i in range(count):
                src = f"decoder.transformer_blocks.{i}"
                for mine, theirs in (("qkv", "attn.to_qkv"), ("out", "attn.to_out"), ("w1", "ff.w1"), ("w2", "ff.w2")):
                    tensors[f"{i}.{mine}.qdata"] = _read(f, f"{src}.{theirs}.weight")
                    tensors[f"{i}.{mine}.qscale"] = _read(f, f"{src}.{theirs}.weight_scale").float().reshape(-1)
                    tensors[f"{i}.{mine}.bias"] = _read(f, f"{src}.{theirs}.bias").to(torch.bfloat16)
                for name in ("norm1", "norm2"):
                    tensors[f"{i}.{name}"] = _read(f, f"{src}.{name}.weight").to(torch.bfloat16)
                for name in ("scale1", "scale2"):
                    tensors[f"{i}.{name}"] = _read(f, f"{src}.{name}").to(torch.bfloat16)
            for mine, theirs in (("xe_w", "decoder.x_embedder.weight"), ("xe_b", "decoder.x_embedder.bias"),
                                 ("norm_out_w", "decoder.norm_out.weight"), ("norm_out_b", "decoder.norm_out.bias"),
                                 ("proj_w", "decoder.proj_out.weight"), ("proj_b", "decoder.proj_out.bias"),
                                 ("pqc_w", "post_quant_conv.weight"), ("pqc_b", "post_quant_conv.bias")):
                self.small[mine] = _read(f, theirs).float()
            self.small["registers"] = _read(f, "decoder.register_tokens").reshape(-1, SOL_DIM).to(torch.bfloat16)
        names = list(tensors)
        self.weights = [{} for _ in range(count)]
        for name, view in zip(names, pack_pinned(tensors, names, "VAE decoder")):
            i, rest = name.split(".", 1)
            self.weights[int(i)][rest] = view
        self.num_layers = count

    @torch.no_grad()
    def decode(self, latents, mean, std, rounding="truncate", progress=None):
        """Normalized H3 latents [1, 24, T, H, W] -> video [3, frames, 16 H, 16 W] in RAM (uint8 with a rounding, else
        floats in [-1, 1])."""
        z = latents[0].float().cpu()
        C, T, H, W = z.shape
        z = z * std.float().cpu().view(-1, 1, 1, 1) + mean.float().cpu().view(-1, 1, 1, 1)
        z = (torch.einsum("oc,cthw->thwo", self.small["pqc_w"].reshape(C, C), z) + self.small["pqc_b"]).contiguous()  # post_quant_conv
        nb = -(-T // SOL_BLOCK)
        starts = [0]
        for t in range(T):
            starts.append(starts[-1] + (1 if t % SOL_BLOCK == 0 else SOL_PATCH_T))
        video = torch.empty(3, starts[-1], H * SOL_PATCH, W * SOL_PATCH, dtype=torch.float32 if rounding is None else torch.uint8)
        split = SOL_SPLIT["on"] and len(DEVICES) > 1 and nb >= 2
        cut = min(nb - 1, max(1, round(T / (2 * SOL_BLOCK)))) if split else nb  # GPU 0: native blocks 0..cut-1
        job = {"z": z, "T": T, "H": H, "W": W, "nb": nb, "cut": cut, "video": video, "starts": starts, "rounding": rounding,
               "progress": progress}
        latents_per_gpu = (T - SOL_BLOCK * cut + SOL_BLOCK) if split else T
        x_bytes = max(latents_per_gpu, SOL_BLOCK * cut) * H * W * SOL_DIM * 2
        frees = [torch.cuda.mem_get_info(d)[0] for d in DEVICES if d.type == "cuda"]
        if frees:  # windows per attention call: room for the scores even if SDPA takes its plain path
            SOL_WINDOW_GROUP["n"] = int(min(16, max(2, (min(frees) - x_bytes - (2 << 30)) // (400 * 2**20))))
        if split:
            n = (SOL_BLOCK * H * W + SOL_GLOBALS) * SOL_DIM * 2
            try:
                host = pinned_empty_exact(n)
            except Exception:
                host = torch.empty(n, dtype=torch.uint8).pin_memory()
            job["host"] = host.view(torch.bfloat16)
            _run_on_both(self._decode_part, [(job,)] * len(DEVICES))
        else:
            with _on(DEVICES[-1]):
                self._decode_part(len(DEVICES) - 1, job)
        return video

    def _decode_part(self, s, job):
        """GPU s's share of a decode: its native blocks through every layer, then their frames."""
        dev, D, P = DEVICES[s], SOL_DIM, SOL_PATCH
        T, H, W, nb, cut, host = job["T"], job["H"], job["W"], job["nb"], job["cut"], job.get("host")
        split, edge = cut < nb, SOL_BLOCK * cut
        if not split:
            lo, hi, windows = 0, T, None
        elif s == 0:
            lo, hi, windows = 0, edge, (0, cut)
        else:  # from the start of block cut - 1, which GPU 1's first shifted windows reach
            lo, hi, windows = edge - SOL_BLOCK, T, (cut, 1 << 30)
        small = {k: v.to(dev, non_blocking=True) for k, v in self.small.items()}
        x = torch.empty(hi - lo, H, W, D, dtype=torch.bfloat16, device=dev)
        for t in range(lo, hi):  # every latent -> one token (FP32, as the reference)
            x[t - lo] = torch.nn.functional.linear(job["z"][t].to(dev).view(H * W, -1), small["xe_w"], small["xe_b"]).view(H, W, D)
        g = torch.cat((small["registers"], small["registers"].new_zeros(1, D)))[None].repeat(nb, 1, 1)
        slot = _sol_layer_buffers(torch.nn.Module(), dev)
        targets = dict(slot.named_buffers())
        for index in range(self.num_layers):
            shifted = index % 2 == 1
            if split and index > 0:
                # shifted layer: GPU 1's first windows reach block cut - 1, finished by GPU 0 (its 5 latents and global
                # tokens). Unshifted layer: GPU 0 takes back the 2 latents of block cut - 1 that GPU 1 finished.
                (a, b), sender = ((edge - SOL_BLOCK, edge), 0) if shifted else ((edge - SOL_T_SHIFT, edge), 1)
                part = x[a - lo:b - lo].view(-1)
                n = part.numel()
                if s == sender:
                    host[:n].copy_(part)
                    if shifted:
                        host[n:n + SOL_GLOBALS * D].copy_(g[cut - 1].reshape(-1))
                _TP_BARRIER.wait()
                if s != sender:
                    part.copy_(host[:n])
                    if shifted:
                        g[cut - 1].view(-1).copy_(host[n:n + SOL_GLOBALS * D])
            for name, target in targets.items():
                target.copy_(self.weights[index][name], non_blocking=True)
            self._layer(slot, shifted, x, g, T, lo, windows)
            if job["progress"] is not None and s == len(DEVICES) - 1:
                job["progress"](index + 1, self.num_layers + 1)
        del g, slot, targets
        last_shift = SOL_T_SHIFT if (self.num_layers - 1) % 2 == 1 else 0
        out_lo, out_hi = (0, T) if not split else (0, edge - last_shift) if s == 0 else (edge - last_shift, T)
        for t in range(out_lo, out_hi):  # the latents this GPU finished in the last layer -> frames
            y = torch.nn.functional.layer_norm(x[t - lo].reshape(-1, D).float(), (D,), small["norm_out_w"], small["norm_out_b"], 1e-5)
            px = torch.nn.functional.linear(y, small["proj_w"], small["proj_b"]).view(H, W, 3, SOL_PATCH_T, P, P)
            del y
            if t % SOL_BLOCK == 0:  # the first latent of a native block keeps its last frame
                px = px[:, :, :, SOL_PATCH_T - 1:]
            n = px.shape[3]
            frames = _sol_pixels(px.permute(2, 3, 0, 4, 1, 5).reshape(3, n, H * P, W * P), job["rounding"])
            job["video"][:, job["starts"][t]:job["starts"][t] + n] = frames.cpu()
            del px, frames
        del x
        if job["progress"] is not None and s == len(DEVICES) - 1:
            job["progress"](self.num_layers + 1, self.num_layers + 1)


def _read(f, key):
    return f.get_tensor(key)


def build_variant(lora_file, module_file, lora_runtime, head, base):
    """{name: CPU tensor} of every GPU buffer an adapter fills. Without a module file, blocks 46-49 get the base model's
    own weights back (from `base`), and the LoRA file's pairs on them."""
    out = {}
    with safe_open(lora_file, framework="pt", device="cpu") as f:
        keys = set(f.keys())
        groups = {}
        for i in range(NUM_BLOCKS):
            for attr in ("attn.out_proj", "mlp.fc1", "mlp.fc2"):
                name = f"blocks.{i}.{attr}"
                if f"{name}.lora_A.weight" in keys:
                    groups[name] = (_read(f, f"{name}.lora_A.weight"), _read(f, f"{name}.lora_B.weight"))
        for (name, s), (a, b) in lora_runtime.split(groups).items():
            out[f"lora.{name}.{s}.A"], out[f"lora.{name}.{s}.B"] = a, b
        del groups
        # AdaLN curve projections and the output head: the base plus the adapter's difference (FP32)
        for prefix in [f"blocks.{i}.adaln_proj.linear" for i in range(NUM_BLOCKS)] + ["final_layer.adaln_proj.linear"]:
            delta = _read(f, f"{prefix}.lora_B.weight").float() @ _read(f, f"{prefix}.lora_A.weight").float()
            out[f"{prefix}.weight"] = (base[f"{prefix}.weight"] + delta).contiguous()
            out[f"{prefix}.bias"] = (base[f"{prefix}.bias"] + _read(f, f"{prefix}.diff_b").float()).contiguous()
        for prefix in ("final_layer.video_out", "final_layer.audio_out"):
            out[f"{prefix}.weight"] = base[f"{prefix}.weight"] + _read(f, f"{prefix}.diff").float()
            out[f"{prefix}.bias"] = base[f"{prefix}.bias"] + _read(f, f"{prefix}.diff_b").float()
        out["final_layer.norm.weight"] = (base["final_layer.norm.weight"].float()
                                          + _read(f, "final_layer.norm.diff").float()).to(base["final_layer.norm.weight"].dtype)
    if module_file is None:
        for i in PI_REPLACED_BLOCKS:
            for attr in ("attn.out_proj", "mlp.fc1", "mlp.fc2"):
                for s in range(len(DEVICES)):
                    for part in ("qdata", "qscale"):
                        out[f"blocks.{i}.{attr}.{s}.{part}"] = base[f"blocks.{i}.{attr}.{s}.{part}"]
        return out
    with safe_open(module_file, framework="pt", device="cpu") as f:
        for i in PI_REPLACED_BLOCKS:
            block = transformer.blocks[i]
            for short, attr in (("out", "attn.out_proj"), ("fc1", "mlp.fc1"), ("fc2", "mlp.fc2")):
                data = _read(f, f"blocks.{i}.{attr}.weight")
                scale = _read(f, f"blocks.{i}.{attr}.weight_scale").float().reshape(-1)
                for s in range(len(DEVICES)):
                    index = block.splits[s][short]
                    if short == "fc1":  # rows: SwiGLU gate rows, then value rows
                        out[f"blocks.{i}.{attr}.{s}.qdata"] = data.index_select(0, index).contiguous()
                        out[f"blocks.{i}.{attr}.{s}.qscale"] = scale.index_select(0, index).contiguous()
                    else:  # input columns on 256-wide rotation-group boundaries
                        out[f"blocks.{i}.{attr}.{s}.qdata"] = data[:, index].contiguous()
                        out[f"blocks.{i}.{attr}.{s}.qscale"] = scale.contiguous()
            del data
        targets = dict(head.named_buffers())
        for mine, theirs in head.source_names().items():
            value = _read(f, theirs)
            target = targets[mine]
            value = value.reshape(target.shape).to(target.dtype) if value.dtype != torch.int8 else value.reshape(target.shape)
            out[f"dwd.{mine}"] = value.contiguous()
    return out


def gpu_buffers(lora_parts_gpu, head):
    """The GPU tensors each adapter tensor is copied into, by name."""
    gpu = dict(lora_parts_gpu)
    for i in range(NUM_BLOCKS):
        for s, shard in enumerate(transformer.blocks[i].shards):
            if s == len(DEVICES) - 1:
                gpu[f"blocks.{i}.adaln_proj.linear.weight"] = shard.adaln_proj.linear.weight
                gpu[f"blocks.{i}.adaln_proj.linear.bias"] = shard.adaln_proj.linear.bias
            else:  # the other GPU's copy of the same AdaLN: filled from the same tensor
                gpu[f"blocks.{i}.adaln_proj.linear.weight@{s}"] = shard.adaln_proj.linear.weight
                gpu[f"blocks.{i}.adaln_proj.linear.bias@{s}"] = shard.adaln_proj.linear.bias
    fl = transformer.final_layer
    gpu["final_layer.adaln_proj.linear.weight"] = fl.adaln_proj.linear.weight
    gpu["final_layer.adaln_proj.linear.bias"] = fl.adaln_proj.linear.bias
    gpu["final_layer.video_out.weight"], gpu["final_layer.video_out.bias"] = fl.video_out.weight, fl.video_out.bias
    gpu["final_layer.audio_out.weight"], gpu["final_layer.audio_out.bias"] = fl.audio_out.weight, fl.audio_out.bias
    gpu["final_layer.norm.weight"] = fl.norm.weight
    for i in PI_REPLACED_BLOCKS:
        for s, shard in enumerate(transformer.blocks[i].shards):
            for attr, module in (("attn.out_proj", shard.attn.out_proj), ("mlp.fc1", shard.mlp.fc1), ("mlp.fc2", shard.mlp.fc2)):
                gpu[f"blocks.{i}.{attr}.{s}.qdata"], gpu[f"blocks.{i}.{attr}.{s}.qscale"] = module.qdata, module.qscale
    for name, value in head.named_buffers():
        gpu[f"dwd.{name}"] = value
    return gpu


def _source_name(name):
    return name.split("@", 1)[0]


def pack_pinned(tensors, names, label="pi-H3 adapters"):
    """One exact-size pinned buffer holding the tensors (in `names` order), returned as views."""
    sizes = [-(-tensors[n].numel() * tensors[n].element_size() // 64) * 64 for n in names]
    try:
        buffer = pinned_empty_exact(sum(sizes))
    except Exception:
        buffer = torch.empty(sum(sizes), dtype=torch.uint8).pin_memory()
    views, offset = [], 0
    for name, size in zip(names, sizes):
        t = tensors[name]
        view = buffer[offset:offset + t.numel() * t.element_size()].view(t.dtype).view(t.shape)
        view.copy_(t)
        views.append(view)
        offset += size
    PINNED_BYTES[label] = PINNED_BYTES.get(label, 0) + sum(sizes)
    return views


PI = {"active": None, "store": {}, "names": None, "gpu": None, "late": set()}


def pack_mode(mode, tensors, share_with=()):
    """The mode's tensors in pinned RAM as [(name, view)]; tensors equal to the `share_with` modes' are not stored twice."""
    names = sorted(tensors)
    shared = {}
    for other in share_with:
        theirs = dict(PI["store"].get(other, ()))
        for n in names:
            if n not in shared and n in theirs and theirs[n].shape == tensors[n].shape \
                    and theirs[n].dtype == tensors[n].dtype and torch.equal(theirs[n], tensors[n]):
                shared[n] = theirs[n]
    own = [n for n in names if n not in shared]
    views = dict(zip(own, pack_pinned(tensors, own)))
    PI["store"][mode] = [(n, shared[n] if n in shared else views[n]) for n in names]


def use_adapter(mode):
    """Copy the adapter of `mode` into the GPU buffers (a no-op when it is already there)."""
    if PI["active"] == mode:
        return
    for name, cpu in PI["store"][mode]:
        for target in PI["gpu"][name]:
            with torch.cuda.device(target.device) if target.is_cuda else _on(target.device):
                target.copy_(cpu, non_blocking=True)
    for dev in DEVICES:
        if dev.type == "cuda":
            torch.cuda.synchronize(dev)
    lora_runtime.off = set() if mode in VAE_MODES else PI["late"]
    PI["active"] = mode
# <<< section


BAR.start("pi", "Loading: pi-H3 adapters, VAE decoder")
lora_runtime = LoraRuntime(transformer)
dwd_head = DWDHead(MAIN_DEV, transformer.hidden_size)
# the base tensors the adapters change (read before any adapter is copied in): AdaLN curves, output head, blocks 46-49
_fl = transformer.final_layer
_base = {"final_layer.adaln_proj.linear.weight": _fl.adaln_proj.linear.weight, "final_layer.adaln_proj.linear.bias": _fl.adaln_proj.linear.bias,
         "final_layer.video_out.weight": _fl.video_out.weight, "final_layer.video_out.bias": _fl.video_out.bias,
         "final_layer.audio_out.weight": _fl.audio_out.weight, "final_layer.audio_out.bias": _fl.audio_out.bias,
         "final_layer.norm.weight": _fl.norm.weight}
for _i in range(NUM_BLOCKS):
    _lin = transformer.blocks[_i].shards[-1].adaln_proj.linear
    _base[f"blocks.{_i}.adaln_proj.linear.weight"], _base[f"blocks.{_i}.adaln_proj.linear.bias"] = _lin.weight, _lin.bias
_base = {k: v.detach().float().cpu() if k != "final_layer.norm.weight" else v.detach().cpu() for k, v in _base.items()}
for _i in PI_REPLACED_BLOCKS:
    for _s, _shard in enumerate(transformer.blocks[_i].shards):
        for _attr, _m in (("attn.out_proj", _shard.attn.out_proj), ("mlp.fc1", _shard.mlp.fc1), ("mlp.fc2", _shard.mlp.fc2)):
            _base[f"blocks.{_i}.{_attr}.{_s}.qdata"] = _m.qdata.detach().cpu().clone()
            _base[f"blocks.{_i}.{_attr}.{_s}.qscale"] = _m.qscale.detach().cpu().clone()


def _register(tensors):
    """GPU buffers (and LoRA hooks) for the adapter tensors that have none yet. Returns the new LoRA names."""
    lora = {(k[5:].rsplit(".", 2)[0], int(k.rsplit(".", 2)[1])): (tensors[k], tensors[k[:-1] + "B"])
            for k in tensors if k.startswith("lora.") and k.endswith(".A") and k not in (PI["gpu"] or {})}
    gpu = lora_runtime.attach(lora)
    if PI["gpu"] is None:
        gpu, PI["gpu"] = gpu_buffers(gpu, dwd_head), {}
    for n, t in gpu.items():
        PI["gpu"].setdefault(_source_name(n), []).append(t)
    PI["names"] = sorted(PI["gpu"])
    return {name for name, _ in lora}


def _check(tensors, optional):
    missing = [n for n in PI["names"] if n not in tensors and not optional(n)]
    if missing:
        raise SystemExit(f"❌ The pi-H3 adapter has no tensor for {len(missing)} layer(s), e.g. {missing[:3]}")
    for n, value in tensors.items():
        for t in PI["gpu"].get(n, ()):
            if tuple(t.shape) != tuple(value.shape):
                raise SystemExit(f"❌ pi-H3 adapter tensor {n}: shape {tuple(value.shape)}, expected {tuple(t.shape)}")


_late = tuple(f"lora.blocks.{i}." for i in PI_REPLACED_BLOCKS)
PI["late"] = set()
# the regular-decode adapters share their blocks 0-45 with the DWD adapter of the same step count, and the base
# weights of blocks 46-49 with each other
for _mode, (_lora_file, _module_file), _share in (("dwd2", PI_FILES[2], ()), ("dwd4", PI_FILES[4], ()),
                                                  ("vae2", (PI_VAE_LORAS[2], None), ("dwd2",)),
                                                  ("vae4", (PI_VAE_LORAS[4], None), ("dwd4", "vae2"))):
    _tensors = build_variant(os.path.join(MODELS_DIR, _lora_file), _module_file and os.path.join(MODELS_DIR, _module_file),
                             lora_runtime, dwd_head, _base)
    _new = _register(_tensors)
    if _mode in VAE_MODES:  # LoRAs on blocks 46-49: only these adapters have them
        PI["late"] |= set().union(*(lora_runtime.ids[name] for name in _new))
    _check(_tensors, (lambda n: n.startswith("dwd.")) if _mode in VAE_MODES else (lambda n: n.startswith(_late)))
    pack_mode(_mode, _tensors, share_with=_share)
    del _tensors
    gc.collect()
del _base
sol_vae = SolVAEDecoder(os.path.join(MODELS_DIR, SOL_VAE_FILE))
gc.collect()
use_adapter("dwd2")
lora_runtime.strength = 1.0
BAR.finish()

# ==== DWD: the last step's video rows -> pixels, in place of the VAE decoder ====
# The transformer wrapper (install_hooks) recognises the last step by its sigma and arms the capture; the final layer
# then hands its input rows to the head's AdaLN (FP16 features kept on GPU 1 when there is room, otherwise in RAM).
# The pipeline's VAE decode call returns the head's frames instead of decoding the latent, or, in the VAE-decoder mode,
# the frames of Sol-H3-VAE's decoder on both GPUs.
DWD = {"last_sigma": None, "capture": False, "t": None, "grid": None, "features": None, "seconds": 0.0}
_orig_final_forward = transformer.final_layer.forward


def _final_forward(x_list, temb, video_segment, audio_segment):
    if DWD["capture"]:
        DWD["capture"] = False
        v0, v1, _ = video_segment
        hidden = x_list[0]
        rows = v1 - v0
        need = rows * hidden.shape[-1] * 2
        on_gpu = hidden.is_cuda and torch.cuda.mem_get_info(hidden.device)[0] > 2 * need + (1 << 30)
        out = torch.empty(rows, hidden.shape[-1], dtype=torch.float16, device=hidden.device if on_gpu else "cpu")
        DWD["features"] = dwd_head.features(hidden[v0:v1], DWD["t"], out)
    return _orig_final_forward(x_list, temb, video_segment, audio_segment)


transformer.final_layer.forward = _final_forward


def _dwd_decode(latents, uint8_rounding=None):
    """Stands in for the video VAE's decode: the frames of the DWD head, uint8 [1, 3, frames, H, W] in RAM."""
    features, grid = DWD["features"], DWD["grid"]
    DWD.update(features=None)
    if features is None:
        raise RuntimeError("No DWD features were captured at the last step")
    PEAKS["denoise"] = [torch.cuda.max_memory_allocated(d) for d in DEVICES]  # the DiT phase ends here
    for d in DEVICES:
        torch.cuda.reset_peak_memory_stats(d)
    t0 = time.time()
    BAR.start("dwd", "Pixel head (DWD)", SOL_LAYERS + 1, "layers")

    def progress(done, total):
        BAR.update(done, total)
        report(0.88 + 0.07 * done / total, f"🎞️ Pixel head {done}/{total}")

    with torch.cuda.device(MAIN_DEV):
        gc.collect()
        torch.cuda.empty_cache()
        holder = [features]
        del features
        video = dwd_head.decode(holder, grid, progress)
    BAR.finish()
    DWD["seconds"] = time.time() - t0
    return video[None]


def _vae_decode(latents, uint8_rounding=None):
    """The VAE-decoder mode: the latent through Sol-H3-VAE's decoder on both GPUs, uint8 [1, 3, frames, H, W] in RAM."""
    PEAKS["denoise"] = [torch.cuda.max_memory_allocated(d) for d in DEVICES]  # the DiT phase ends here
    for d in DEVICES:
        torch.cuda.reset_peak_memory_stats(d)
    t0 = time.time()
    BAR.start("vae", "VAE decoding", sol_vae.num_layers + 1, "layers")

    def progress(done, total):
        BAR.update(done, total)
        report(0.88 + 0.07 * done / total, f"🎞️ VAE decoding {done}/{total}")

    gc.collect()
    for d in DEVICES:
        with torch.cuda.device(d):
            torch.cuda.empty_cache()
    video = sol_vae.decode(latents, pipeline.vae._latents_mean, pipeline.vae._latents_std, uint8_rounding, progress)
    BAR.finish()
    DWD["seconds"] = time.time() - t0
    return video[None]


def _decode(latents, uint8_rounding=None):
    return _vae_decode(latents, uint8_rounding) if PI["active"] in VAE_MODES else _dwd_decode(latents, uint8_rounding)


pipeline.vae.decode = _decode

# ==== Text encoder + VAE encoder + audio VAE: MMGP on GPU 1, unloaded before denoising ====
# The video VAE only encodes the first / last frames; its decoder is never used (the DWD head or Sol-H3-VAE's decoder
# makes the frames), so its weights are dropped from RAM.
for _p in list(pipeline.video_decoder.parameters()) + list(pipeline.video_decoder.buffers()):
    try:
        _p.data = torch.empty(0, dtype=_p.dtype)
    except Exception:
        pass
pipeline.audio_vae._convertWeightsFloatTo = None  # keep the audio VAE in FP32 (Turing has no BF16 convs)
aux_pipe = {
    "text_encoder": pipeline.text_encoder.language_model,
    "vision_encoder": pipeline.text_encoder.visual,
    "video_encoder": pipeline.video_encoder,
    "audio_vae": pipeline.audio_vae,
}
if TE_DTYPE == torch.float32:
    pipeline.text_encoder.language_model._convertWeightsFloatTo = None
    pipeline.text_encoder.visual._convertWeightsFloatTo = None
# The text encoder (~14 GB), the VAE encoder and the audio VAE are pinned in RAM: Kaggle's disk reads at ~180 MB/s, so a
# memory-mapped copy that dropped out of the page cache would cost seconds per video before anything could start.
_pin = ["text_encoder", "video_encoder", "audio_vae"]
_profile_args = dict(profile_no=4, quantizeTransformer=False, extraModelsToQuantize=None, convertWeightsFloatTo=torch.bfloat16,
                     asyncTransfers=True, verboseLevel=-1, budgets={"text_encoder": 1500, "vision_encoder": 0, "*": 3000})
BAR.start("pin", "Loading: pinning RAM")
try:
    aux_offload = offload.profile(aux_pipe, pinnedMemory=_pin, **_profile_args)
except Exception as e:
    print(f"⚠️ Pinning failed ({e}); the text encoder streams from regular RAM")
    aux_offload = offload.profile(aux_pipe, pinnedMemory=False, **_profile_args)
pipeline.set_offload_handoff(aux_offload, None)  # the pipeline unloads these before handing the GPU to the DiT
BAR.finish()
if hasattr(torch._C, "_host_emptyCache"):
    torch._C._host_emptyCache()

gc.collect()
for _dev in DEVICES:
    with torch.cuda.device(_dev):
        torch.cuda.empty_cache()
_free0, _free1 = torch.cuda.mem_get_info(0)[0], torch.cuda.mem_get_info(1)[0]

PEAKS = {}
_orig_audio_decode = pipeline.audio_vae.decode


def _audio_decode(*args, **kwargs):
    BAR.start("audio", "Audio decoding")
    report(0.96, "🔊 Decoding the audio")
    return _orig_audio_decode(*args, **kwargs)


pipeline.audio_vae.decode = _audio_decode


# ==== Attention: exact FP16 SDPA, or Comfy Kitchen's INT8 attention ====
# Attention is the largest part of a denoising step (~15k tokens at 480p 5 s, ~37k at 768p 5 s).
# Fast (default): Comfy Kitchen's INT8 attention, a compiled CUDA kernel with Turing (SM 7.5) code. Q and K get a Hadamard
# rotation (same dot products, smaller outliers) and go to INT8, V to INT8 per channel, the softmax weights to unsigned
# INT8, with the softmax itself in FP32; the T4's INT8 tensor cores run at twice the FP16 rate. It is offered only when a
# startup test on both GPUs passes and shows it faster, and every INT8 run still compares it with exact attention on the
# user's video in its first step (blocks 1 and 26, GPU 0), switching back to exact by itself if they differ too much.
# Exact: PyTorch SDPA in FP16 (its memory-efficient kernel; the T4 has no flash kernel).
# Wan2GP's own SageAttention path is Triton INT8 code, which Kaggle's Triton cannot compile for the T4.
# >>> section:attention
ATTENTION_EXACT, ATTENTION_INT8 = "Exact (FP16)", "Fast (Kitchen INT8)"
ATTENTION_MAX_DIFF = 0.08  # relative L2 difference from exact attention above which an INT8 run switches back to exact
INT8_MIN_TOKENS = 2048     # INT8 only for the DiT's long sequences (the text token refiner stays exact)
INT8_DEVICE_TYPES = ("cuda",)
ATTENTION = {"mode": "sdpa", "int8": None, "int8_dtype": None, "int8_contiguous": False, "int8_speedup": None, "int8_reason": "",
             "checks": set(), "fallback": None}
offload.shared_state["_attention"] = "sdpa"
import shared.attention as _w2g_attention
_sdpa_attention = _w2g_attention.pay_attention  # Wan2GP's own (SDPA here), never the wrapper below
_ATTENTION_CHECK_LOCK = threading.Lock()


def _int8_attention(q, k, v):
    """Kitchen INT8 attention on [batch, tokens, heads, dim] tensors (H3's layout), output in q's dtype and layout."""
    out_dtype, dtype = q.dtype, ATTENTION["int8_dtype"] or q.dtype
    args = [t.transpose(1, 2) if t.dtype == dtype else t.transpose(1, 2).to(dtype) for t in (q, k, v)]
    if ATTENTION["int8_contiguous"]:
        args = [t.contiguous() for t in args]
    return ATTENTION["int8"].int8_attention(*args).transpose(1, 2).to(out_dtype)


def _relative_diff(out, ref):
    """||out - ref|| / ||ref|| in FP32, by slices of tokens (no full-size FP32 copies)."""
    num = torch.zeros((), dtype=torch.float32, device=out.device)
    den = torch.zeros((), dtype=torch.float32, device=out.device)
    for a in range(0, out.shape[1], 4096):
        o, r = out[:, a:a + 4096].float(), ref[:, a:a + 4096].float()
        num += (o - r).square().sum()
        den += r.square().sum()
    return (num / den.clamp_min(1e-30)).sqrt().item()


def _check_int8_attention(q, k, v):
    """INT8 vs exact attention on this block's real Q / K / V. Returns the output to use (exact if INT8 is too far off)."""
    dev = q.device
    if dev.type == "cuda":
        free = torch.cuda.mem_get_info(dev)[0] + torch.cuda.memory_reserved(dev) - torch.cuda.memory_allocated(dev)
        if free < 8 * q.numel() * q.element_size() + (512 << 20):  # the exact reference needs copies of Q / K / V
            return _int8_attention(q, k, v)
    ref = _sdpa_attention([q.clone(), k.clone(), v.clone()])
    out = _int8_attention(q, k, v)
    diff = _relative_diff(out, ref)
    if not math.isfinite(diff) or diff > ATTENTION_MAX_DIFF:
        ATTENTION.update(mode="sdpa", fallback=diff)
        return ref
    return out


def pay_attention_tp(qkv_list, recycle_q=False, **kwargs):
    """H3's attention call: Kitchen INT8 for the DiT's long sequences when the run asks for it, else Wan2GP's (SDPA)."""
    q = qkv_list[0]
    if (ATTENTION["mode"] != "int8" or kwargs or q.dim() != 4 or q.shape[1] < INT8_MIN_TOKENS
            or q.device.type not in INT8_DEVICE_TYPES):
        return _sdpa_attention(qkv_list, recycle_q=recycle_q, **kwargs)
    q, k, v = qkv_list
    qkv_list.clear()
    check = False
    if q.device == DEVICES[0] and ATTENTION["checks"]:
        with _ATTENTION_CHECK_LOCK:  # one check per listed block, whichever thread gets there first
            check = TP_STATE["block"] in ATTENTION["checks"]
            ATTENTION["checks"].discard(TP_STATE["block"])
    return _check_int8_attention(q, k, v) if check else _int8_attention(q, k, v)


_h3_transformer.pay_attention = pay_attention_tp


def _probe_case(dev, fused):
    """INT8 vs SDPA difference on random BF16 data in one of H3's two layouts: fused (Q and K are views of one fused
    projection output, V a copy: the normal path) or separate contiguous tensors (long-sequence mode)."""
    g = torch.Generator(device=dev).manual_seed(0)
    tokens, heads, dim = 3000, 4, 128
    if fused:
        qkv = torch.randn(1, tokens, 3 * heads * dim, device=dev, generator=g).to(torch.bfloat16)
        q, k, v = (t.view(1, tokens, heads, dim) for t in qkv.split(heads * dim, dim=-1))
        v = v.clone()
    else:
        q, k, v = (torch.randn(1, tokens, heads, dim, device=dev, generator=g).to(torch.bfloat16) for _ in range(3))
    ref = _sdpa_attention([q.clone(), k.clone(), v.clone()])
    return _relative_diff(_int8_attention(q, k, v), ref)


def _probe_int8_attention():
    """Kitchen INT8 attention on both GPUs against SDPA on random data (catches a kernel that does not load, a dtype it
    does not take on Turing, or strides it does not follow), then a speed test at a DiT-like size."""
    try:
        import comfy_kitchen
        if not all(comfy_kitchen.int8_attention_is_available(dev) for dev in DEVICES):
            ATTENTION["int8_reason"] = "not available on this GPU"
            return
        ATTENTION["int8"] = comfy_kitchen
        # BF16 straight from the projections and the projections' own strides first; FP16 / contiguous copies otherwise
        for dtype, contiguous in ((torch.bfloat16, False), (torch.bfloat16, True), (torch.float16, False), (torch.float16, True)):
            ATTENTION.update(int8_dtype=dtype, int8_contiguous=contiguous)
            try:
                worst = 0.0
                for dev in DEVICES:
                    with _on(dev), torch.inference_mode():
                        worst = max(worst, _probe_case(dev, True), _probe_case(dev, False))
                if math.isfinite(worst) and worst < 0.1:
                    break
                ATTENTION["int8_reason"] = f"self-test differed {worst:.1%} from SDPA"
            except Exception as e:
                ATTENTION["int8_reason"] = f"self-test failed: {str(e).splitlines()[0][:120] if str(e) else type(e).__name__}"
            ATTENTION["int8_dtype"] = None
        if ATTENTION["int8_dtype"] is None:
            ATTENTION["int8"] = None
            return
        ATTENTION["int8_reason"] = ""
        with _on(MAIN_DEV), torch.inference_mode():
            q, k, v = (torch.randn(1, 8192, 28, 128, device=MAIN_DEV).to(torch.bfloat16) for _ in range(3))
            timings = []
            for run in (lambda: _sdpa_attention([q, k, v]), lambda: _int8_attention(q, k, v)):
                for _ in range(2):
                    run()
                start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
                start.record()
                for _ in range(3):
                    run()
                end.record()
                end.synchronize()
                timings.append(start.elapsed_time(end))
            ratio = timings[0] / max(timings[1], 1e-3)
            del q, k, v
        if ratio < 1.05:  # offered only when it is actually faster on these GPUs
            ATTENTION.update(int8=None, int8_dtype=None, int8_reason=f"not faster than exact attention here: {ratio:.2f}x")
        else:
            ATTENTION["int8_speedup"] = ratio
    except Exception as e:
        ATTENTION.update(int8=None, int8_dtype=None, int8_reason=f"unavailable: {str(e).splitlines()[0][:120] if str(e) else type(e).__name__}")
    finally:
        for dev in DEVICES:
            with _on(dev):
                torch.cuda.empty_cache()


def attention_speedup(choice):
    return (ATTENTION["int8_speedup"] or 1.0) if choice == ATTENTION_INT8 and ATTENTION["int8"] is not None else 1.0
# <<< section


BAR.start("attention", "Loading: testing Kitchen INT8 attention", 1, "test")
_probe_int8_attention()
BAR.finish()
ATTENTION_CHOICES = ([ATTENTION_INT8] if ATTENTION["int8"] is not None else []) + [ATTENTION_EXACT]
ATTENTION_DEFAULT = ATTENTION_CHOICES[0]
ATTENTION_LABEL = "Kitchen INT8 attention" if ATTENTION["int8"] is not None else "SDPA attention"

# ==== Drop the checkpoint files from the page cache ====
# Loading reads ~40 GB of checkpoints and Linux keeps what fits of them cached. That cache is reclaimable, but Kaggle's
# RAM meter counts it. Everything is pinned or on the GPUs now, so the cached copies are useless.
def drop_file_cache(*roots):
    for root in roots:
        for dirpath, _, files in os.walk(root, followlinks=True):
            for name in files:
                try:
                    fd = os.open(os.path.join(dirpath, name), os.O_RDONLY)
                    try:
                        os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
                    finally:
                        os.close(fd)
                except (OSError, AttributeError):
                    pass


def _ram_report():
    vm = psutil.virtual_memory()
    cache = getattr(vm, "cached", 0) + getattr(vm, "buffers", 0)
    return f"{_ram_in_use()} used by the engine | {_gb(cache)} file cache (reclaimable, Kaggle's meter counts it)"


drop_file_cache(MODELS_DIR)

# ==== Sizes, time estimates ====
# >>> section:io
FPS = 24
# H3 generates 17n + 5 frames at 24 fps (5.2 to 14.4 s, as the official demo)
DURATIONS = {f"{n / FPS:.1f} s · {n} frames": n for n in (124, 175, 243, 294, 345)}
DEFAULT_DURATION = next(iter(DURATIONS))
RESOLUTIONS = {"480p (0.4 MP) ⚡ fastest": 0.4e6, "576p (0.6 MP)": 0.6e6, "768p (1 MP, native) 🐢 slower": 1.0e6}
DEFAULT_RESOLUTION = next(iter(RESOLUTIONS))
ASPECTS = ("16:9", "9:16", "1:1", "4:3", "3:4", "21:9")
STEP_CHOICES = {"⚡ 2 steps (fastest)": "dwd2", "💎 4 steps (more detail)": "dwd4",
                "🎨 2 steps + VAE decoder (cleaner frames)": "vae2", "🎨 4 steps + VAE decoder (best quality)": "vae4"}
DEFAULT_STEPS = next(iter(STEP_CHOICES))


def canvas_size(aspect, pixels, multiple=32):
    """(width, height) of the aspect ratio at about `pixels`, both rounded up to the multiple (1 MP 16:9 -> 1344 x 768,
    H3's native size), as the official demo."""
    a, b = (float(v) for v in aspect.split(":"))
    return (int(math.ceil(math.sqrt(pixels * a / b) / multiple) * multiple),
            int(math.ceil(math.sqrt(pixels * b / a) / multiple) * multiple))


def fit_image(image, w, h):
    """A first / last frame at the canvas size: scaled to cover and centre-cropped (ImageOps.fit, as the official demo)."""
    from PIL import ImageOps
    return ImageOps.fit(image.convert("RGB"), (w, h), Image.Resampling.LANCZOS)


def video_rows(w, h, frames, images):
    """Packed sequence rows of one forward: text, the target video, each first / last frame latent, audio."""
    per_frame = (h // 32) * (w // 32)
    video = h3_pipeline.video_latent_frames(frames) * per_frame
    audio = (round((frames + 1) / FPS * h3_pipeline.AUDIO_LATENT_FPS) + 5) * 2
    return 700 + video + images * per_frame + audio


def sol_tokens(w, h, frames):
    return h3_pipeline.video_latent_frames(frames) * (h // 16) * (w // 16)


def step_seconds(rows, attention_speedup=1.0):
    """Seconds per step on Kaggle T4 x2 with exact attention: 24 x^2 + 7 x + 6, x = rows / 16k. The quadratic term is
    attention, divided by the speedup of the chosen attention."""
    x = rows / 16384
    return 24 * x * x / attention_speedup + 7 * x + 6


def needed_bytes(rows, long_video=False, head_groups=4):
    """Working VRAM per GPU during denoising: activations per packed row (half the heads per GPU) + scratch. The normal
    path peaks at about 70 KB per row. Long-video mode keeps the residual stream, the attention output and one group of
    heads' Q / K / V at full length: fewer, larger groups need more memory and run faster."""
    if not long_video:
        per_row = 72
    else:
        per_row = 18 + 72 // head_groups if OVERLAP["on"] else 48
    return rows * per_row * 1024 + 700 * 1024**2


def denoise_plan(rows, free):
    """(long-video mode, head groups): the normal path when it fits, else long-video mode with the fewest groups that fit."""
    if needed_bytes(rows) <= free:
        return False, 1
    for groups in (2, 4):
        if needed_bytes(rows, True, groups) <= free:
            return True, groups
    return True, 4


def dwd_bytes(w, h, frames):
    """Working VRAM of the pixel head on GPU 1: the Sol tokens (BF16) and the per-block attention buffers."""
    per_block = 5 * (h // 16) * (w // 16)
    return sol_tokens(w, h, frames) * SOL_DIM * 2 + per_block * SOL_DIM * 2 * 20 + 900 * 1024**2


def vae_bytes(w, h, frames):
    """Working VRAM of the VAE decoder per GPU: about half the clip's tokens + one native block (BF16), the per-block
    attention buffers and a layer's weights."""
    per_block = 5 * (h // 16) * (w // 16)
    return (sol_tokens(w, h, frames) // 2 + per_block) * SOL_DIM * 2 + per_block * SOL_DIM * 2 * 20 + 900 * 1024**2


def decode_seconds(w, h, frames, mode):
    """The pixel head (8 Sol layers on GPU 1) or the VAE decoder (24 Sol layers, split over both GPUs)."""
    tokens = sol_tokens(w, h, frames)
    return 3 + tokens * 1.5e-4 if mode in VAE_MODES else 2 + tokens * 9e-5


def encode_seconds(w, h, images):
    """The frames' VAE encode + the prompt encoding, which shows the frames to Qwen3-VL (one token per 32x32 pixels)."""
    tokens = 400 + images * round(h / 32) * round(w / 32)
    return 10 + 0.01 * tokens + 10 * images


def total_seconds(w, h, frames, images, mode, attention):
    rows = video_rows(w, h, frames, images)
    long_mode, _ = denoise_plan(rows, min(_free0, _free1))
    step = step_seconds(rows, attention_speedup(attention)) * 1.02 * (1.4 if long_mode else 1.0)
    save = 3 + 4e-8 * w * h * frames
    return encode_seconds(w, h, images) + PI_MODES[mode] * step + decode_seconds(w, h, frames, mode) + save
# <<< section


# ==== Live progress (also polled by the page over plain HTTP, which survives tunnels dropping Gradio's event stream) ====
PROGRESS = {"active": False, "fraction": 0.0, "text": "", "started": 0.0, "finished": 0.0, "video": None, "message": "", "final_n": 0}


def report(fraction, text):
    PROGRESS.update(active=True, fraction=float(fraction), text=text)


def free_for_denoise():
    aux_offload.unload_all()
    gc.collect()
    frees = []
    for dev in DEVICES:
        with torch.cuda.device(dev):
            torch.cuda.empty_cache()
        frees.append(torch.cuda.mem_get_info(dev)[0])
    return frees


def save_mp4(video, path, audio=None, audio_rate=AUDIO_RATE, fps=FPS):
    """Streams a uint8 [3, T, H, W] video to ffmpeg 16 frames at a time, with the generated stereo audio."""
    _, t, h, w = video.shape
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{w}x{h}", "-r", str(fps), "-i", "-"]
    wav = None
    if audio is not None:
        import soundfile as sf
        wav = path[:-4] + ".wav"
        sf.write(wav, np.asarray(audio, dtype=np.float32), int(audio_rate))
        cmd += ["-i", wav, "-map", "0:v:0", "-map", "1:a:0", "-c:a", "aac", "-b:a", "192k"]
    cmd += ["-c:v", "libx264", "-preset", "veryfast", "-crf", "17", "-pix_fmt", "yuv420p", "-movflags", "+faststart",
            "-t", f"{t / fps:.3f}", path]  # the video's length (the audio carries a short tail past the last frame)
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE)
    try:
        for start in range(0, t, 16):
            frames = video[:, start:start + 16]
            if frames.dtype != torch.uint8:
                frames = frames.float().clamp(-1, 1).add(1).mul(127.5).round().to(torch.uint8)
            proc.stdin.write(frames.permute(1, 2, 3, 0).contiguous().numpy().tobytes())
        proc.stdin.close()
    except BrokenPipeError:
        pass
    code = proc.wait()
    if wav and os.path.exists(wav):
        os.remove(wav)
    if code != 0:
        raise RuntimeError(f"ffmpeg failed with exit code {code}")


# ==== Encoded first / last frames are cached between videos ====
# Re-running with the same pictures (another seed, a tweaked prompt) reuses their VAE latents (the encode is
# deterministic: same input, same latents). The prompt encoding has its own cache in Wan2GP.
ENCODE_CACHE = collections.OrderedDict()


def attention_warning(int8):
    """A status line only when an INT8 run had to switch back to exact attention."""
    if not int8 or ATTENTION["fallback"] is None:
        return ""
    return "⚠️ Fast attention was not accurate enough on this video, so it was finished with exact attention"


def install_hooks(pipe):
    """The encode cache, the DWD capture at the last step and the switch to per-GPU threads after the first transformer
    pass, on this pipeline."""
    orig_encode_video = pipe._encode_video
    orig_forward = pipe.transformer.forward

    def cached_encode_video(video, keep_all_latents=False):
        if video.is_cuda:
            return orig_encode_video(video, keep_all_latents=keep_all_latents)
        data = video.contiguous()
        digest = hashlib.blake2b(memoryview(data.view(torch.uint8).reshape(-1).numpy()), digest_size=16).hexdigest()
        vae = pipe.vae
        key = (id(pipe), digest, str(video.dtype), tuple(video.shape), bool(keep_all_latents), getattr(vae, "use_tiling", None),
               getattr(vae, "tile_sample_min_height", None), getattr(vae, "tile_sample_min_width", None))
        if key in ENCODE_CACHE:
            ENCODE_CACHE.move_to_end(key)
            return ENCODE_CACHE[key].clone()
        latent = orig_encode_video(video, keep_all_latents=keep_all_latents)
        ENCODE_CACHE[key] = latent.clone()
        while len(ENCODE_CACHE) > 8:
            ENCODE_CACHE.popitem(last=False)
        return latent

    def forward(video_x, audio_x, sigma_video, sigma_audio, *args, **kwargs):
        sigma = float(sigma_video.flatten().max())
        if DWD["last_sigma"] is not None and abs(sigma - DWD["last_sigma"]) < 1e-4:  # the last step: hand its video rows to the head
            _, _, latent_t, latent_h, latent_w = video_x.shape
            DWD.update(capture=True, t=1.0 - sigma, grid=(latent_t, latent_h // 2, latent_w // 2))
        out = orig_forward(video_x, audio_x, sigma_video, sigma_audio, *args, **kwargs)
        TP_THREADS["warm"] = True  # the first pass of the session (one thread) has compiled every kernel on both GPUs
        return out

    pipe._encode_video = cached_encode_video
    pipe.transformer.forward = forward


if "pipeline" in globals():
    install_hooks(pipeline)


@torch.inference_mode()
def run_generation(kwargs, steps):
    pipeline._interrupt = False
    t0 = time.time()
    timings = {"encode": 0.0, "denoise": 0.0, "decode": 0.0}
    phase = {"name": "prep", "t": time.time(), "step_t": time.time()}

    def callback(step, latent=None, is_start=False, override_num_inference_steps=None, progress_unit=None,
                 progress_title=None, **kw):
        now = time.time()
        if progress_unit == "layers":  # Qwen3-VL prompt encoding
            phase.setdefault("prompt_t", now)
            total = override_num_inference_steps or 50
            BAR.start("text", "Prompt encoding", total, "layers")
            BAR.update(max(0, step + 1), total)
            report(0.05 + 0.10 * min(1.0, (step + 1) / total), f"📝 Encoding the prompt {max(0, step + 1)}/{total}")
            return
        if progress_unit == "tiles":  # VAE encoding of the first / last frames
            total = override_num_inference_steps or 1
            done = min(total, max(0, step + 1))
            BAR.start("encode", "Encoding frames", total, "tiles")
            BAR.update(done, total)
            report(0.02 + 0.03 * done / total, f"🖼️ Encoding the first / last frames {done}/{total}")
            return
        if is_start:
            timings["encode"] = now - t0
            timings["prompt"] = now - phase.get("prompt_t", now)
            phase.update(name="denoise", t=now, step_t=now, total=override_num_inference_steps or steps)
            BAR.start("denoise", "Denoising", phase["total"], "steps")
            report(0.15, f"🎬 Denoising step 0/{phase['total']}")
            return
        total = phase.get("total", steps)
        dt = now - phase["step_t"]
        phase["step_t"] = now
        done = step + 1
        BAR.update(done, total)
        report(0.15 + 0.72 * done / total, f"🎬 Denoising step {done}/{total} ({dt:.0f}s/step)")

    def set_status(text):
        if "Decod" in str(text) and phase["name"] == "denoise":
            timings["denoise"] = time.time() - phase["t"]
            phase.update(name="decode", t=time.time())

    report(0.01, "⚡ Preparing...")
    BAR.start("prep", "Preparing inputs")
    try:
        result = pipeline.generate(callback=callback, set_progress_status=set_status, **kwargs)
    finally:
        BAR.finish()
    if result is None:
        return None, timings
    if phase["name"] == "decode":
        timings["decode"] = time.time() - phase["t"]
    elif phase["name"] == "denoise":
        timings["denoise"] = phase["step_t"] - phase["t"]
        timings["decode"] = time.time() - phase["step_t"]
    timings["total"] = time.time() - t0
    return result, timings


def on_main_gpu(fn):
    """CUDA's current device is per thread: Gradio workers start on GPU 0. Run handlers with GPU 1 current, the device
    the sampler, text encoder and VAEs use for "cuda"."""
    @functools.wraps(fn)
    def wrapped(*args, **kwargs):
        with torch.cuda.device(MAIN_DEV):
            return fn(*args, **kwargs)
    return wrapped


def _pil(value):
    if value is None:
        return None
    if isinstance(value, str):
        value = Image.open(value)
    return value.convert("RGB")


@on_main_gpu
def generate_video(prompt, first_image, last_image, resolution, aspect, duration, steps_label, seed, attention=ATTENTION_EXACT):
    PROGRESS.update(active=True, fraction=0.0, text="Starting...", started=time.time(), finished=0.0, video=None, message="")
    try:
        prompt = (prompt or "").strip()
        if not prompt:
            return None, "❌ Write a prompt first: describe the whole video, its sound and any spoken lines."
        first, last = _pil(first_image), _pil(last_image)
        mode = STEP_CHOICES.get(steps_label, "dwd2")
        steps, vae = PI_MODES[mode], mode in VAE_MODES
        w, h = canvas_size(aspect if aspect in ASPECTS else "16:9", RESOLUTIONS.get(resolution, 0.4e6))
        frames = DURATIONS.get(duration, 124)
        images = int(first is not None) + int(last is not None)
        if seed is None or int(seed) < 0:
            seed = random.randint(0, 2**31 - 1)
        seed = int(seed)

        rows = video_rows(w, h, frames, images)
        frees = free_for_denoise()
        long_video, head_groups = denoise_plan(rows, min(frees))
        LONG_VIDEO.update(on=long_video, head_groups=head_groups)
        decode_ok = vae_bytes(w, h, frames) < min(frees) if vae else dwd_bytes(w, h, frames) < frees[1]
        if needed_bytes(rows, long_video, head_groups) > min(frees) or not decode_ok:
            return None, (f"❌ {w}x{h} for {frames / FPS:.1f} s needs more GPU memory than is free ({_gb(frees[0])} / "
                          f"{_gb(frees[1])}). Pick a shorter duration or a lower resolution.")
        int8 = attention == ATTENTION_INT8 and ATTENTION["int8"] is not None
        estimate = total_seconds(w, h, frames, images, mode, attention)
        use_adapter(mode)
        sigmas = pi_sigmas(steps)
        DWD.update(last_sigma=None if vae else sigmas[-1], capture=False, features=None, seconds=0.0)
        kwargs = dict(input_prompt=prompt, frame_num=frames, height=h, width=w, fps=FPS, shift=PI_SHIFT, sampling_steps=steps,
                      seed=seed, sample_solver="euler", guide_phases=1, custom_sigmas=[sigmas], video_prompt_type="",
                      audio_prompt_type="", batch_size=1)
        if first is not None:
            kwargs["image_start"] = h3_pipeline._pil_to_video(fit_image(first, w, h))
        if last is not None:
            kwargs["image_end"] = h3_pipeline._pil_to_video(fit_image(last, w, h))
        what = {0: "text to video", 1: "first frame" if first is not None else "last frame", 2: "first + last frames"}[images]
        print(f"🎬 {w}x{h} | {frames / FPS:.1f}s | {what} | {steps} steps{' + VAE decoder' if vae else ''}"
              f"{' | fast attention' if int8 else ''} | seed {seed} "
              f"| est. ~{_fmt_secs(estimate)}")
        sys.stdout.flush()

        for dev in DEVICES:
            torch.cuda.reset_peak_memory_stats(dev)
        PEAKS.clear()
        # INT8 attention is checked against exact attention on this video in the first step (blocks 1 and 26, GPU 0)
        ATTENTION.update(mode="int8" if int8 else "sdpa", checks={0, NUM_BLOCKS // 2} if int8 else set(), fallback=None)
        result, timings = run_generation(kwargs, steps)
        if result is None:
            return None, "⏹️ Generation stopped."
        report(0.98, "💾 Saving MP4...")
        out_path = os.path.join(OUTPUT_DIR, f"pih3_{steps}step{'_vae' if vae else ''}_{time.strftime('%Y%m%d_%H%M%S')}_seed{seed}.mp4")
        BAR.start("save", "Saving MP4")
        save_mp4(result["x"], out_path, result.get("audio"), result.get("audio_sampling_rate", AUDIO_RATE))
        BAR.finish()
        total = timings.get("total", 0.0)
        per_step = timings["denoise"] / steps
        out_h, out_w = result["x"].shape[-2:]
        status = "\n".join([
            f"✅ Done in {_fmt_secs(total)}  (pi-H3 {steps} steps{' + VAE decoder' if vae else ''}, {what})",
            f"⚡ Frames + prep {timings['encode'] - timings.get('prompt', 0.0):.0f}s | "
            + (f"Prompt {timings['prompt']:.0f}s" if timings.get("prompt", 0.0) >= 0.5 else "Prompt cached")
            + f" | Denoising {timings['denoise']:.0f}s ({per_step:.1f}s/step) | "
            f"{'VAE decoder' if vae else 'Pixel head'} {DWD['seconds']:.0f}s | Audio + rest {max(0.0, timings['decode'] - DWD['seconds']):.0f}s",
            f"📐 {out_w}x{out_h} | {result['x'].shape[1]} frames @ 24 fps ({frames / FPS:.1f}s) | 🎲 Seed {seed} | "
            + ("⚡ Fast attention" if int8 and ATTENTION["fallback"] is None else "exact attention"),
            f"🧠 Peak VRAM per GPU: "
            + (f"denoise {_gb(PEAKS['denoise'][0])} / {_gb(PEAKS['denoise'][1])}, " if "denoise" in PEAKS else "")
            + f"{'VAE decoder' if vae else 'pixel head'} {_gb(torch.cuda.max_memory_allocated(0))} / {_gb(torch.cuda.max_memory_allocated(1))}",
            f"📁 {out_path}",
        ] + [line for line in (attention_warning(int8),) if line])
        print(f"✅ Done in {_fmt_secs(total)} ({per_step:.0f}s/step) → {os.path.basename(out_path)}")
        drop_file_cache(OUTPUT_DIR)
        sys.stdout.flush()
        return out_path, status
    except torch.OutOfMemoryError:
        traceback.print_exc()
        return None, "❌ Out of GPU memory. Pick a lower resolution or a shorter duration and try again."
    except Exception as e:
        traceback.print_exc()
        if "CUDA error" in str(e):  # e.g. an illegal memory access: the GPU context is unusable from here on
            print("❌ The GPUs are in an error state. Re-run Step 2 to restart the engine (the models reload in ~4 min).")
            return None, f"❌ GPU error: {e}\nRe-run Step 2 to restart the engine."
        return None, f"❌ Error: {e}"
    finally:
        ATTENTION.update(mode="sdpa", checks=set())
        DWD.update(last_sigma=None, capture=False, features=None)
        BAR.abort()
        PROGRESS["active"] = False
        for key in _tp_cache:  # last hidden state / rope / temb copies held on both GPUs
            _tp_cache[key] = None
        gc.collect()
        for dev in DEVICES:
            with torch.cuda.device(dev):
                torch.cuda.empty_cache()


# ==== Generation as a background job ====
# The Generate button only starts a job and returns; the page script polls /aiq_progress over plain HTTP and clicks a
# hidden button to fetch the result when a new one is ready, so a reloaded page or a dropped connection loses nothing.
JOB = {"id": 0, "done": 0, "running": False, "video": None, "status": "Ready."}


def _job_worker(job_id, args):
    try:
        video, status = generate_video(*args)
    except BaseException as e:  # generate_video reports its own errors; this is a last resort
        video, status = None, f"❌ Error: {e}"
    JOB.update(running=False, done=job_id, video=video, status=status)
    PROGRESS.update(active=False, finished=time.time(), video=os.path.basename(video) if video else None,
                    message=status.splitlines()[0] if status else "", final_n=job_id)


def start_generation(*args):
    if JOB["running"]:
        return "⏳ A video is already being generated. It appears here as soon as it is ready."
    JOB.update(id=JOB["id"] + 1, running=True)
    PROGRESS.update(active=True, fraction=0.0, text="Starting...", started=time.time(), finished=0.0, video=None, message="")
    threading.Thread(target=_job_worker, args=(JOB["id"], args), daemon=True).start()
    return ("🚀 Started. Progress shows at the top of the page and the video appears below when it is ready. "
            "Reloading the page or a dropped connection does not stop the job.")


def deliver_result():
    if JOB["running"]:
        return gr.skip(), f"⏳ {PROGRESS['text'] or 'Working...'}"
    if not JOB["done"]:
        return gr.skip(), JOB["status"]
    return JOB["video"], JOB["status"]


def stop_generation():
    if not JOB["running"]:
        return "Nothing is running."
    pipeline._interrupt = True
    return "⏹️ Stopping after the current step..."


def latest_video_path():
    videos = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.mp4")), key=os.path.getmtime)
    return videos[-1] if videos else None


def load_latest_video():
    path = latest_video_path()
    if not path:
        return None, f"No videos in {OUTPUT_DIR} yet."
    return path, f"📁 Loaded the latest video: {path}"


def size_info(resolution, aspect, duration, steps_label, attention=ATTENTION_EXACT, first_image=None, last_image=None):
    w, h = canvas_size(aspect if aspect in ASPECTS else "16:9", RESOLUTIONS.get(resolution, 0.4e6))
    frames = DURATIONS.get(duration, 124)
    images = int(first_image is not None) + int(last_image is not None)
    seconds = total_seconds(w, h, frames, images, STEP_CHOICES.get(steps_label, "dwd2"), attention)
    return f"ℹ️ {w}x{h}, {frames} frames ({frames / FPS:.1f} s): about {_fmt_secs(seconds)} per video."


# ==== Official examples (the pi-H3 demo's own prompts, pictures, sizes, durations and steps) ====
PIH3_SPACE_REVISION = "c8569fd3f0a2774b096c9c23437da3e5266dfaee"  # Lakonik/pi-H3 Space, 22 Nov 2026
EXAMPLE_FILES = ["examples.json", "distracted_boyfriend.webp", "woman_cat_meme_first.webp", "woman_cat_meme_last.webp"]


def ensure_examples():
    """The demo's example files (Step 1 downloads them; fetched here if they are missing)."""
    missing = [f for f in EXAMPLE_FILES if not os.path.exists(os.path.join(EXAMPLES_DIR, f))]
    if not missing:
        return True
    try:
        from huggingface_hub import hf_hub_download
        os.makedirs(EXAMPLES_DIR, exist_ok=True)
        for f in missing:
            path = hf_hub_download("Lakonik/pi-H3", f"demo/examples/minimax_h3/{f}", repo_type="space",
                                   revision=PIH3_SPACE_REVISION, local_dir=os.path.join(EXAMPLES_DIR, ".hf"))
            shutil.copy(path, os.path.join(EXAMPLES_DIR, f))
        return True
    except Exception as e:
        print(f"⚠️ The official examples could not be downloaded ({str(e).splitlines()[0][:120]}); the UI shows none")
        return False


def _example_resolution(megapixels):
    return min(RESOLUTIONS, key=lambda k: abs(RESOLUTIONS[k] - megapixels * 1e6))


def _example_duration(seconds):
    n = min(max(round((seconds * 24 - 5) / 17), 7), 20) * 17 + 5  # the demo's rounding to 17n + 5 frames
    return next((k for k, v in DURATIONS.items() if v == n), DEFAULT_DURATION)


def official_example_rows():
    """Rows for gr.Examples: [prompt, first frame, last frame, resolution, aspect, duration, steps]. The seed stays random."""
    if not ensure_examples():
        return [], []
    with open(os.path.join(EXAMPLES_DIR, "examples.json"), encoding="utf-8") as f:
        examples = json.load(f)

    def local(url):
        return None if not url else os.path.join(EXAMPLES_DIR, url.rsplit("/", 1)[-1])

    steps_of = {2: next(k for k, v in STEP_CHOICES.items() if v == "dwd2"), 4: next(k for k, v in STEP_CHOICES.items() if v == "dwd4")}
    rows, labels = [], []
    for i, ex in enumerate(examples[:3]):
        steps = steps_of.get(2 if int(ex.get("steps", 2)) < 3 else 4, DEFAULT_STEPS)
        resolution = _example_resolution(float(ex.get("megapixels", 1.0)))
        duration = _example_duration(float(ex.get("duration", 5.0)))
        aspect = ex.get("aspect_ratio", "16:9")
        if i == 0:  # the text-to-video example also as a quick 480p test
            quick = [ex["prompt"], None, None, DEFAULT_RESOLUTION, aspect, duration, steps]
            w, h = canvas_size(aspect, RESOLUTIONS[DEFAULT_RESOLUTION])
            rows.append(quick)
            labels.append(f"⚡ {ex['label']}: quick test, 480p {DURATIONS[duration] / FPS:.0f} s "
                          f"(~{_fmt_secs(total_seconds(w, h, DURATIONS[duration], 0, STEP_CHOICES[steps], ATTENTION_DEFAULT))})")
            continue
        first, last = local(ex.get("in_image")), local(ex.get("in_last_image"))
        images = int(first is not None) + int(last is not None)
        w, h = canvas_size(aspect, RESOLUTIONS[resolution])
        took = total_seconds(w, h, DURATIONS[duration], images, STEP_CHOICES[steps], ATTENTION_DEFAULT)
        icon = "🖼️" if images == 1 else "🎞️"
        what = "first frame" if images == 1 else "first + last frames"
        rows.append([ex["prompt"], first, last, resolution, aspect, duration, steps])
        labels.append(f"{icon} {ex['label']}: official, {what}, 1 MP {DURATIONS[duration] / FPS:.0f} s (~{_fmt_secs(took)})")
    return rows, labels


ensure_examples()


# ==== Warm-up: one short step at startup ====
# The first transformer pass of a session runs on one thread and sets up every kernel on both GPUs; a 1-step clip
# here does that (and runs the pixel head and audio once), so the first video you generate is as fast as the next ones.
# Around it, two checks: the FP16 text encoder against the plain path on the warm-up prompt, and the overlapped GPU
# exchange against the plain exchange on the warm-up clip's first block (captured during the clip).
WARM_PROMPT = "integrated_multimodal_description: [Shot 1] A plain grey studio wall in soft daylight."


def _standardized_diff(out, ref):
    """Mean |out - ref| in units of each channel's spread over the tokens (how the DiT sees the prompt states)."""
    out, ref = out.float().reshape(-1, out.shape[-1]), ref.float().reshape(-1, ref.shape[-1])
    spread = ref.std(dim=0, unbiased=False).clamp_min(1e-6) if ref.shape[0] > 1 else ref.abs().clamp_min(1e-6)
    return float(((out - ref).abs() / spread).mean())


@torch.inference_mode()
def check_te_fp16():
    if not TE_FP16["on"]:
        return
    ok = False
    try:
        with torch.cuda.device(MAIN_DEV):
            encoder = pipeline.text_encoder
            fast = encoder.encode(WARM_PROMPT, [], pipeline.device, pipeline.dtype)[0].float()
            TE_FP16["on"] = False
            ref = encoder.encode(WARM_PROMPT, [], pipeline.device, pipeline.dtype)[0].float()
            ok = bool(torch.isfinite(fast).all()) and _standardized_diff(fast, ref) < 0.02
            TE_FP16["diff"] = _standardized_diff(fast, ref)
    except Exception:
        ok = False
    TE_FP16["on"] = ok


def _capture_first_block():
    """Keep the first block's inputs of the next transformer pass (the warm-up clip) for check_overlap."""
    block = transformer.blocks[0]
    original = block.forward

    def forward(x_list, temb, segments, rope, residual_signature_elements=0):
        if OVERLAP["capture"]:
            OVERLAP.update(capture=False, sample=(x_list[0].clone(), temb, list(segments), rope))
        return original(x_list, temb, segments, rope, residual_signature_elements)

    block.forward = forward
    OVERLAP["capture"] = True
    return lambda: block.__dict__.pop("forward", None)


@torch.inference_mode()
def check_overlap():
    sample, OVERLAP["sample"] = OVERLAP["sample"], None
    if sample is None or not OVERLAP["on"]:
        return
    x, temb, segments, rope = sample
    block = transformer.blocks[0]

    def run():
        for key in _tp_cache:
            _tp_cache[key] = None
        out = block([x.clone()], temb, segments, rope).float()
        for dev in DEVICES:
            torch.cuda.synchronize(dev)
        return out

    ok = False
    try:
        with torch.cuda.device(MAIN_DEV):
            TP_THREADS["warm"] = True
            OVERLAP["on"] = False
            ref = run()
            OVERLAP["on"] = True
            diffs = []
            for _ in range(3):
                out = run()
                diffs.append(float((out - ref).norm() / ref.norm().clamp_min(1e-12)) if bool(torch.isfinite(out).all()) else float("inf"))
            OVERLAP["diff"] = max(diffs)
            ok = OVERLAP["diff"] < 5e-3
    except Exception:
        ok = False
    OVERLAP["on"] = ok
    for key in _tp_cache:
        _tp_cache[key] = None


@torch.inference_mode()
def warm_up():
    BAR.start("warm", "Loading: warm-up clip")
    release = _capture_first_block()
    try:
        check_te_fp16()
        with torch.cuda.device(MAIN_DEV):
            ATTENTION.update(mode="int8" if ATTENTION["int8"] is not None else "sdpa", checks=set(), fallback=None)
            free_for_denoise()
            pipeline._interrupt = False
            sigma = pi_sigmas(2)[0]
            DWD.update(last_sigma=sigma, capture=False, features=None)
            pipeline.generate(input_prompt=WARM_PROMPT,
                              frame_num=39, height=480, width=864, fps=FPS, shift=PI_SHIFT, sampling_steps=1, seed=0,
                              sample_solver="euler", guide_phases=1, custom_sigmas=[[sigma]], video_prompt_type="",
                              audio_prompt_type="", batch_size=1, callback=None)
    except Exception as e:
        print(f"⚠️ Warm-up skipped ({str(e).splitlines()[0][:120] if str(e) else type(e).__name__}); the first video takes a little longer")
    finally:
        release()
        OVERLAP["capture"] = False
        ATTENTION.update(mode="sdpa", checks=set(), fallback=None)
        DWD.update(last_sigma=None, capture=False, features=None)
        BAR.finish()
        PROGRESS.update(active=False, fraction=0.0, text="")
        for key in _tp_cache:
            _tp_cache[key] = None
        gc.collect()
        for dev in DEVICES:
            with torch.cuda.device(dev):
                torch.cuda.empty_cache()


if os.environ.get("PIH3_WARMUP", "1") == "1":
    warm_up()
    check_overlap()


# ==== Gradio UI (AIQUEST Academy branding) ====
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#aiq-deliver { display: none !important; }
.status-line { text-align: center; color: #6b7280; font-size: 13px; margin: -8px 0 12px 0; }
.brand-footer { text-align: center; color: #6b7280; font-size: 12px; margin-top: 24px; }
"""

BRAND_HTML = """
<div class="brand-header">
  <div class="brand-title">⚡ pi-H3: Video + Audio in 2 to 4 Steps</div>
  <div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> &nbsp;|&nbsp; Kaggle GPU T4 x2 Edition</div>
  <div class="btn-row">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe</a>
    <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
</div>
"""
STATUS_HTML = (f'<div class="status-line">⚡ {INT8_LABEL} &nbsp;|&nbsp; {ATTENTION_LABEL} &nbsp;|&nbsp; MiniMax H3 20B '
               f'tensor-parallel on 2 x T4 &nbsp;|&nbsp; pi-H3 DWD: frames without a VAE decoder &nbsp;|&nbsp; Powered by Wan2GP</div>')
FOOTER_HTML = ('<div class="brand-footer">⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved<br>'
               'Models: <a href="https://huggingface.co/Lakonik/pi-H3" target="_blank">pi-H3</a> on '
               '<a href="https://huggingface.co/MiniMaxAI/MiniMax-H3" target="_blank">MiniMax H3</a> '
               '(MiniMax H3 Community License) · label results as AI-generated.</div>')

PROGRESS_HEAD = """
<script>
(function () {
  // Progress overlay fed by plain HTTP polling: it keeps working when a tunnel drops Gradio's event stream. When a new
  // result is ready it clicks the hidden deliver button once, which fetches the video (no timer re-rendering the page).
  var box = null, fill = null, label = null, hideAt = 0, lastFinal = null;
  function build() {
    box = document.createElement("div");
    box.style.cssText = "position:fixed;top:12px;left:50%;transform:translateX(-50%);z-index:99999;width:min(560px,92vw);" +
      "background:#1f2937;border-radius:12px;box-shadow:0 8px 24px rgba(0,0,0,.25);padding:10px 14px;display:none;" +
      "font-family:Inter,sans-serif;color:#fff;font-size:13px";
    label = document.createElement("div");
    label.style.cssText = "margin-bottom:6px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis";
    var track = document.createElement("div");
    track.style.cssText = "height:8px;background:rgba(255,255,255,.18);border-radius:6px;overflow:hidden";
    fill = document.createElement("div");
    fill.style.cssText = "height:100%;width:0;background:linear-gradient(135deg,#667eea 0%,#764ba2 100%);transition:width .6s ease";
    track.appendChild(fill); box.appendChild(label); box.appendChild(track); document.body.appendChild(box);
  }
  async function tick() {
    try {
      var r = await fetch("aiq_progress?t=" + Date.now(), {cache: "no-store"});
      var p = await r.json();
      if (!box && document.body) build();
      if (lastFinal === null) { lastFinal = p.final_n; }
      var deliver = document.getElementById("aiq-deliver");
      if (deliver && p.final_n && p.final_n !== lastFinal && !p.active) { lastFinal = p.final_n; deliver.click(); }
      if (box) {
        if (p.active) {
          box.style.display = "block"; hideAt = 0;
          fill.style.width = Math.round(p.fraction * 100) + "%";
          label.textContent = "⏳ " + p.text + " · " + Math.round(p.elapsed) + "s";
        } else if (box.style.display === "block") {
          if (!hideAt) {
            fill.style.width = "100%";
            label.textContent = (p.video ? "✅ Done in " : "⚠️ Finished after ") + Math.round(p.elapsed) + "s ";
            if (p.video) {
              var link = document.createElement("a");
              link.href = "aiq_video?t=" + Date.now(); link.target = "_blank";
              link.style.cssText = "color:#c7d2fe;font-weight:600";
              link.textContent = "· 🎬 open / download the video";
              label.appendChild(link);
            } else if (p.message) { label.textContent += "· " + p.message; }
            hideAt = Date.now() + 60000;
          } else if (Date.now() > hideAt) { box.style.display = "none"; }
        }
      }
    } catch (e) {}
    setTimeout(tick, 1000);
  }
  tick();
})();
</script>
"""

HOW_TO = ("**How it works:** describe the whole clip (picture, sound, spoken lines) and press **🎬 Generate**. Add a "
          "**first frame** to animate a picture, a **last frame** to land on one, or both. ⚡ 2 steps at 480p is the "
          "fastest; 💎 4 steps and 768p give more detail; the 🎨 VAE decoder modes give cleaner frames (🎨 4 steps is the best).")
PROMPT_TIP = ("H3 prompts have sections: `integrated_multimodal_description:` ([Shot 1] what we see, the camera, and spoken "
              "lines as (S1) ... <d>[English] the line</d>), `overall_soundscape:` and `non_diegetic_music:`. "
              "With pictures, start with how they align, e.g. \"For the target video, at 0.00 seconds into the target video, "
              "<Picture 1> (from [Shot 1]) is fully referenced.\" The examples below show the full format.")

with gr.Blocks(theme=gr.themes.Soft(), css=CSS, head=PROGRESS_HEAD, title="pi-H3 | AIQUEST Academy") as demo:
    gr.HTML(BRAND_HTML)
    gr.HTML(STATUS_HTML)
    gr.Markdown(HOW_TO)

    with gr.Row():
        with gr.Column(scale=3):
            prompt = gr.Textbox(label="📝 Prompt (picture, sound and spoken lines)", lines=12,
                                placeholder="integrated_multimodal_description: [Shot 1] ...\n\noverall_soundscape: ...\n\nnon_diegetic_music: ...")
            gr.Markdown(f"<span style='color:#6b7280;font-size:13px'>💡 {PROMPT_TIP}</span>")
        with gr.Column(scale=2):
            steps_ui = gr.Radio(list(STEP_CHOICES), value=DEFAULT_STEPS, label="🚀 Steps",
                                info="2 and 4 steps draw the frames in the last step; the VAE decoder modes decode "
                                     "them from the latent for the best quality (a little slower)")
            resolution = gr.Dropdown(list(RESOLUTIONS), value=DEFAULT_RESOLUTION, label="📐 Resolution")
            with gr.Row():
                aspect = gr.Dropdown(list(ASPECTS), value="16:9", label="Aspect ratio")
                duration = gr.Dropdown(list(DURATIONS), value=DEFAULT_DURATION, label="⏱️ Duration")
            size_note = gr.Markdown(size_info(DEFAULT_RESOLUTION, "16:9", DEFAULT_DURATION, DEFAULT_STEPS, ATTENTION_DEFAULT))
            seed = gr.Number(value=-1, precision=0, label="🎲 Seed (-1 = random)")
            attention = gr.Radio(ATTENTION_CHOICES, value=ATTENTION_DEFAULT, label="⚡ Attention",
                                 info=("Fast: Comfy Kitchen INT8 attention, quicker on the T4. Exact: full precision"
                                       if ATTENTION["int8"] is not None else "Exact: full precision (Fast is not available on this GPU)"))

    with gr.Accordion("🖼️ First / last frame (optional): <Picture 1>, <Picture 2>", open=False):
        gr.Markdown("Pictures are cropped to the chosen aspect ratio. Call them `<Picture 1>` (the first one you add) and "
                    "`<Picture 2>` in the prompt.")
        with gr.Row():
            first_image = gr.Image(type="pil", label="First frame", height=220)
            last_image = gr.Image(type="pil", label="Last frame", height=220)

    with gr.Row():
        gen_btn = gr.Button("🎬 Generate", variant="primary", size="lg")
        stop_btn = gr.Button("🛑 Stop", size="lg", elem_id="stop-btn")
        clear_btn = gr.Button("🗑️ Clear", size="lg", elem_id="clear-btn")
    video_out = gr.Video(label="🎬 Result (with audio)", height=480, interactive=False)
    latest_btn = gr.Button("📂 Load latest video (use if the UI lost its connection)")
    status_out = gr.Textbox(label="ℹ️ Status", lines=6, interactive=False)
    deliver_btn = gr.Button("deliver", elem_id="aiq-deliver")  # hidden; clicked by the page script when a result is ready

    _rows, _labels = official_example_rows()
    if _rows:
        gr.Markdown("### 📚 Examples\nThe pi-H3 demo's own prompts, pictures and durations. Click one to fill everything "
                    "above, then press **🎬 Generate**.")
        gr.Examples(_rows, inputs=[prompt, first_image, last_image, resolution, aspect, duration, steps_ui],
                    example_labels=_labels, label=None)
    gr.HTML(FOOTER_HTML)

    _info_inputs = [resolution, aspect, duration, steps_ui, attention, first_image, last_image]
    for _comp in _info_inputs:
        _comp.change(size_info, _info_inputs, size_note, queue=False)
    gen_btn.click(start_generation, inputs=[prompt, first_image, last_image, resolution, aspect, duration, steps_ui, seed, attention],
                  outputs=status_out, queue=False)
    deliver_btn.click(deliver_result, outputs=[video_out, status_out], queue=False, show_progress="hidden")
    stop_btn.click(stop_generation, outputs=status_out, queue=False)
    latest_btn.click(load_latest_video, outputs=[video_out, status_out], queue=False)
    clear_btn.click(lambda: (None, "Ready.", "", None, None, -1), outputs=[video_out, status_out, prompt, first_image, last_image, seed],
                    queue=False)



# ==== Cloudflare quick tunnel: a second public link in case the Gradio share link fails ====
# Started without preexec_fn: that would force a real fork() of this process (Python otherwise uses vfork), and a
# fork has to copy every pinned page at once. Step 2 stops leftover tunnels before each start. The link works once
# cloudflared has registered a connection with Cloudflare's edge and the random *.trycloudflare.com name is in DNS;
# --protocol http2 because Kaggle often blocks QUIC.
def _dns_published(host):
    """Ask Cloudflare's DNS-over-HTTPS directly (no local resolver caches a negative answer). True / False / None."""
    try:
        req = urllib.request.Request(f"https://cloudflare-dns.com/dns-query?name={host}&type=A",
                                     headers={"accept": "application/dns-json"})
        return bool(json.loads(urllib.request.urlopen(req, timeout=5).read()).get("Answer"))
    except Exception:
        return None


def start_cloudflare_tunnel(port):
    import stat
    found = {"url": None, "registered": False, "ready": False}
    binary = "/tmp/cloudflared"
    try:
        if not os.path.exists(binary):
            urllib.request.urlretrieve(
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", binary)
            os.chmod(binary, os.stat(binary).st_mode | stat.S_IEXEC)
        tunnel = subprocess.Popen([binary, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate",
                                   "--protocol", "http2"],
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        import atexit
        atexit.register(tunnel.kill)
    except Exception as e:
        print(f"⚠️ Cloudflare tunnel unavailable: {e}")
        return found

    def _watch():
        for line in tunnel.stdout:  # keep draining so cloudflared never blocks on a full pipe
            match = re.search(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com", line)
            if match and not found["url"]:
                found["url"] = match.group(0)
            if "Registered tunnel connection" in line:
                found["registered"] = True

    def _wait_ready():
        deadline, registered_at = time.time() + 300, None
        while time.time() < deadline:
            if found["registered"] and registered_at is None:
                registered_at = time.time()
            published = _dns_published(found["url"].split("//", 1)[1]) if found["url"] and registered_at else False
            if published or (published is None and time.time() - registered_at > 15):
                found["ready"] = True
                if found.get("late"):  # the ready block was already printed without it
                    print(f"🌐 Cloudflare: {found['url']}")
                    sys.stdout.flush()
                return
            time.sleep(3)
        print("⚠️ The Cloudflare link did not come up within 5 minutes; use the Gradio share link.")

    threading.Thread(target=_watch, daemon=True).start()
    threading.Thread(target=_wait_ready, daemon=True).start()
    return found

cf = start_cloudflare_tunnel(SERVER_PORT)
demo.queue(max_size=20)
demo.launch(server_name="127.0.0.1", server_port=SERVER_PORT, share=True, inline=False, show_error=True,
            prevent_thread_lock=True, quiet=True, ssr_mode=False, max_threads=8, allowed_paths=[OUTPUT_DIR, EXAMPLES_DIR])
from fastapi.responses import FileResponse, JSONResponse
from fastapi.routing import APIRoute


def _progress_route():
    end = PROGRESS["finished"] or time.time()
    elapsed = end - PROGRESS["started"] if PROGRESS["started"] else 0.0
    return JSONResponse({**PROGRESS, "elapsed": elapsed}, headers={"Cache-Control": "no-store"})


def _video_route():
    """The newest video as a plain file (plays in the browser, with a download option): no Gradio involved."""
    path = latest_video_path()
    if not path:
        return JSONResponse({"error": "no video yet"}, status_code=404)
    name = os.path.basename(path)
    return FileResponse(path, media_type="video/mp4", headers={"Content-Disposition": f'inline; filename="{name}"',
                                                               "Cache-Control": "no-store"})


for _path_name, _route in (("/aiq_progress", _progress_route), ("/aiq_video", _video_route)):
    demo.app.router.routes.insert(0, APIRoute(_path_name, _route, methods=["GET"]))  # ahead of Gradio's routes
for _ in range(60):  # give the tunnel up to ~30 s to come up before printing the links
    if cf["ready"]:
        break
    time.sleep(0.5)

print(f"✅ Ready in {time.time() - _t_load:.0f}s | GPU free {_gb(_free0)} / {_gb(_free1)} | RAM {_ram_in_use()}")
if cf["ready"]:
    print(f"🌐 Cloudflare: {cf['url']}")
else:
    cf["late"] = True
    print("🌐 Cloudflare: warming up, the link appears here in a minute (Gradio works now)")
print(f"🔗 Gradio: {getattr(demo, 'share_url', None) or 'unavailable, use the Cloudflare link'}")
print(f"💾 Videos: {OUTPUT_DIR} (latest also at <link>/aiq_video)")
sys.stdout.flush()
while True:  # keep the server process alive; generation logs stream to the notebook cell
    time.sleep(3600)
'''




with open("run_pih3.py", "w", encoding="utf-8") as f:
    f.write(RUN_SCRIPT_CODE)

import json
if os.path.exists("pih3_settings.json"):  # Step 1's choices, for a Step 2 re-run after a kernel restart
    with open("pih3_settings.json") as f:
        for key, value in json.load(f).items():
            os.environ.setdefault(key, value)

# Minimal logs: only the engine's own status lines (they start with these markers) and its live progress line reach
# this cell. Library chatter is dropped; Python tracebacks and crash reports are always shown in full.
MARKERS = ("✅", "❌", "⚠️", "🎬", "🌐", "🔗", "💾", "⏹️", "🛑", "🧹", "⚙️", "⏳", "⠋", "⠙", "⠹", "⠸", "⠼", "⠴", "⠦", "⠧", "⠇", "⠏")
CRASH_START = ("Traceback (most recent call last)", "Fatal Python error", "Thread 0x", "Current thread 0x")
_in_crash = [False]


def _keep(part):
    text = part.strip()
    if not text:  # carriage returns / spaces that clear the live line
        return True
    if text.startswith(CRASH_START):
        _in_crash[0] = True
        return True
    if _in_crash[0]:
        if part[:1] in (" ", "\t") or text.startswith(CRASH_START):
            return True
        _in_crash[0] = False  # the exception line closes the traceback
        return True
    return text.startswith(MARKERS)


env = dict(os.environ, PYTHONWARNINGS="ignore", TQDM_DISABLE="1", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
           TF_CPP_MIN_LOG_LEVEL="3", USE_TF="0", USE_FLAX="0", GRADIO_ANALYTICS_ENABLED="False")

print("🚀 Starting pi-H3...")
import psutil


def stop_old_engine():
    # A previous run's engine (or one left behind by a kernel restart) still holds both GPUs and ~24 GB of RAM.
    stale = []
    for p in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            args, name = p.info["cmdline"] or [], (p.info["name"] or "").lower()
            engine = "python" in name and any(os.path.basename(a) == "run_pih3.py" for a in args[1:3])
            if p.pid != os.getpid() and (engine or name == "cloudflared" or name.startswith("frpc")):
                p.kill()
                stale.append(p)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    psutil.wait_procs(stale, timeout=20)
    if stale:
        print(f"🧹 Stopped {len(stale)} leftover engine process(es): GPU memory and RAM are free again")


stop_old_engine()

# Step 1's downloads leave freed buffers in this kernel process: hand them back to the OS before the engine starts.
import ctypes
import gc
gc.collect()
try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass
_kernel_gb = psutil.Process().memory_info().rss / 1024**3
if _kernel_gb > 2.0:
    print(f"⚠️ This notebook kernel holds {_kernel_gb:.1f} GB of RAM. If the engine runs out of memory: Run → Restart, then run Step 2 only.")


def _die_with_kernel():
    # Linux: the engine gets SIGKILL when this notebook kernel dies or restarts (no orphan holding the GPUs).
    try:
        import signal
        ctypes.CDLL("libc.so.6").prctl(1, signal.SIGKILL)  # PR_SET_PDEATHSIG
    except Exception:
        pass


proc = subprocess.Popen([sys.executable, "-u", "run_pih3.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env,
                        preexec_fn=_die_with_kernel)
import threading
import time
_peak = {"ram": 0.0, "engine": 0.0}


def _watch_ram():  # peak RAM while the engine runs, reported if it dies
    engine = psutil.Process(proc.pid)
    while proc.poll() is None:
        try:
            vm = psutil.virtual_memory()
            _peak["ram"] = max(_peak["ram"], (vm.total - vm.available) / 1024**3)
            _peak["engine"] = max(_peak["engine"], engine.memory_info().rss / 1024**3)
        except Exception:
            pass
        time.sleep(0.5)


threading.Thread(target=_watch_ram, daemon=True).start()
# Read raw bytes so carriage returns survive: the engine redraws its live progress line in place, and a text-mode
# pipe would turn every redraw into a new line.
decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
pending = ""
try:
    while True:
        chunk = os.read(proc.stdout.fileno(), 65536)
        if not chunk:
            break
        pending += decoder.decode(chunk)
        parts = re.split(r"(?<=[\r\n])", pending)
        pending = parts.pop()  # text after the last line break / carriage return
        out = "".join(part for part in parts if _keep(part))
        if pending and pending.strip().startswith(MARKERS):
            out, pending = out + pending, ""  # show the live progress line right away
        if out:
            sys.stdout.write(out)
            sys.stdout.flush()
    sys.stdout.write(pending)
    rc = proc.wait()
    if rc != 0:
        import signal
        how = f"killed by {signal.Signals(-rc).name}" if rc < 0 else f"exit code {rc}"
        hint = (" - the system killed it, which almost always means it ran out of RAM" if rc == -9 else
                " - a crash inside a native library (see the traceback above)" if rc in (-11, -6) else "")
        print(f"\n❌ The engine stopped ({how}){hint}.\n   Peak RAM while it ran: {_peak['ram']:.1f} GB in use of "
              f"{psutil.virtual_memory().total / 1024**3:.1f} GB (engine process {_peak['engine']:.1f} GB).")
except KeyboardInterrupt:
    proc.kill()  # frees the GPUs and RAM right away
    proc.wait()
    print("🛑 Engine stopped. Re-run this cell to start it again.")


---

<div align="center">

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>

---